# ⬡ GeoTorpedo — Downhole Drilling Data Analytics
**Vibration  |  RPM  |  Inclination  |  Temperature**

*ICS Dashboard Theme — v3.2 | Dual-CSV Architecture*


## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys
pkgs = ['scipy', 'plotly', 'ipywidgets', 'kaleido']
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])
print('All dependencies ready.')


## Cell 2 — Select CSV Files (Dual-CSV)

In [ ]:

import os
import ipywidgets as widgets
from IPython.display import display, HTML

# ICS theme constants shared across all cells
ICS = {
    'bg_paper': '#0b0c0e', 'bg_plot': '#111318', 'bg_panel': '#181b22',
    'bg_card':  '#1f232d', 'amber':   '#f59e0b', 'amber_dim':'#92600a',
    'teal':     '#14b8a6', 'red':     '#ef4444', 'green':    '#22c55e',
    'blue':     '#3b82f6', 'purple':  '#8b5cf6', 'orange':   '#f97316',
    'text':     '#e8eaf0', 'muted':   '#8b92a5', 'grid':     '#1f232d',
}

hf_widget = widgets.Text(value='', placeholder='Path to 100 Hz high-frequency CSV',
    description='HF CSV:', layout=widgets.Layout(width='85%'),
    style={'description_width':'70px'})
lf_widget = widgets.Text(value='', placeholder='Path to 1 Hz low-frequency CSV',
    description='LF CSV:', layout=widgets.Layout(width='85%'),
    style={'description_width':'70px'})
load_btn  = widgets.Button(description='Load Files', button_style='warning', icon='folder-open')
status_out = widgets.Output()

HF_PATH = None
LF_PATH = None

def on_load(b):
    global HF_PATH, LF_PATH
    hf = hf_widget.value.strip()
    lf = lf_widget.value.strip()
    with status_out:
        status_out.clear_output()
        ok = True
        for path, label in [(hf,'HF (1000 Hz)'),(lf,'LF (1 Hz)')]:
            if not path:
                print(f'  {label}: no path entered.'); ok = False
            elif not os.path.exists(path):
                print(f'  {label}: not found — {path}'); ok = False
            elif not path.lower().endswith('.csv'):
                print(f'  {label}: not a CSV'); ok = False
            else:
                kb = os.path.getsize(path)/1024
                print(f'  {label}: {os.path.basename(path)}  ({kb:.1f} KB)')
        if ok:
            HF_PATH = hf
            LF_PATH = lf

load_btn.on_click(on_load)
display(hf_widget)
display(lf_widget)
display(load_btn)
display(status_out)

# Direct fallback:
# HF_PATH = "C:\Users\fmata\OneDrive - University of Bath\Academic Coursework Management\Year 3\ME32013 MEng Group Business project\GBDP Personal work and references\ics_output\ICS_SIM_20260401_120000_100Hz.csv"
# LF_PATH = "C:\Users\fmata\OneDrive - University of Bath\Academic Coursework Management\Year 3\ME32013 MEng Group Business project\GBDP Personal work and references\ics_output\ICS_SIM_20260401_120000_1Hz.csv"


## Cell 3 — Load CSV Files & Parse Metadata

In [ ]:
import pandas as pd
import numpy as np
import json, math, os, warnings
warnings.filterwarnings('ignore')

if HF_PATH is None or LF_PATH is None:
    raise ValueError('Run Cell 2 and click Load Files first.')

# ── Column definitions ─────────────────────────────────────────────────────────
HF_WANT = [
    'elapsed_time_s',
    'vib_x_raw_g', 'vib_y_raw_g', 'vib_z_raw_g',
    'vib_x_corr_g','vib_y_corr_g','vib_z_corr_g',
    'vib_magnitude_corr_g',
    'vib_x_rms_g', 'vib_y_rms_g', 'vib_z_rms_g',
    'rpm', 'omega_z_dps',
    # rpm_source, drill_condition, condition_severity intentionally excluded
]
LF_WANT = [
    'elapsed_time_s',
    'inclination_deg', 'toolface_deg', 'inclination_valid', 'inclination_source',
    'temp_internal_c', 'temp_external_c', 'thermal_status',
    'rpm_1hz', 'depth_m',
]

def load_csv_with_meta(path, wanted_cols):
    meta_lines, data_lines, header_found = [], [], False
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for raw in f:
            line = raw.strip()
            if line.startswith('#'):
                meta_lines.append(line.lstrip('#').strip())
            else:
                data_lines.append(line)

    # Parse metadata key:value pairs
    meta = {}
    for m in meta_lines:
        if ':' in m:
            k, v = m.split(':', 1)
            meta[k.strip()] = v.strip()
        elif m:
            meta.setdefault('notes', [])
            if isinstance(meta['notes'], list):
                meta['notes'].append(m)

    from io import StringIO
    raw_df = pd.read_csv(StringIO('\n'.join(data_lines)), sep=None, engine='python')
    raw_df.columns = [c.lower().strip().replace(' ', '_') for c in raw_df.columns]

    # Flexible column matching
    keep = {}
    for want in wanted_cols:
        if want in raw_df.columns:
            keep[want] = want
        else:
            for col in raw_df.columns:
                if want.replace('_','') in col.replace('_','') or col.replace('_','') in want.replace('_',''):
                    keep[col] = want
                    break

    available = [c for c in keep.keys() if c in raw_df.columns]
    df = raw_df[available].rename(columns={c: keep[c] for c in available})

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(how='all').reset_index(drop=True)
    df = df.ffill().bfill().fillna(0)
    return meta, df

def detect_rate(df, col='elapsed_time_s'):
    if col not in df.columns: return 1000
    ts = df[col].values
    diffs = np.diff(ts[ts > 0])
    diffs = diffs[diffs > 0]
    if len(diffs) == 0: return 1000
    return max(1, round(1.0 / np.median(diffs)))

print('Loading HF file...')
HF_META, hf_df = load_csv_with_meta(HF_PATH, HF_WANT)
HF_RATE = detect_rate(hf_df)
print(f'  {hf_df.shape[0]:,} rows | {HF_RATE} Hz | cols: {list(hf_df.columns)}')

print('Loading LF file...')
LF_META, lf_df = load_csv_with_meta(LF_PATH, LF_WANT)
LF_RATE = detect_rate(lf_df)
print(f'  {lf_df.shape[0]:,} rows | {LF_RATE} Hz | cols: {list(lf_df.columns)}')

if HF_META:
    print('\nHF Metadata:', {k:v for k,v in HF_META.items() if k != 'notes'})
if LF_META:
    print('LF Metadata:', {k:v for k,v in LF_META.items() if k != 'notes'})
print('CSV load complete.')


## Cell 4 — Metadata Banner

In [ ]:

from IPython.display import display, HTML

def _meta_rows(meta):
    items = [(k,v) for k,v in meta.items() if k != 'notes' and not k.startswith('_')]
    if not items:
        return '<span style="color:#8b92a5;font-size:11px">No metadata found</span>'
    parts = []
    for k,v in items:
        parts.append(
            '<div style="margin:3px 0;display:flex;gap:8px">'
            '<span style="color:#f59e0b;min-width:140px;font-size:10px;'
            'font-family:Space Mono,monospace">' + str(k) + '</span>'
            '<span style="color:#e8eaf0;font-size:11px">' + str(v) + '</span></div>'
        )
    return ''.join(parts)

_hf_rows = _meta_rows(HF_META)
_lf_rows = _meta_rows(LF_META)
_hf_name = os.path.basename(HF_PATH)
_lf_name = os.path.basename(LF_PATH)

META_BANNER_HTML = (
    '<div style="font-family:DM Sans,sans-serif;background:#0b0c0e;'
    'border:1px solid rgba(245,158,11,0.3);border-radius:10px;'
    'overflow:hidden;max-width:1000px;margin-bottom:16px" id="meta-banner">'
    '<div onclick="toggleMeta()" style="background:#1f232d;padding:12px 18px;'
    'border-bottom:1px solid rgba(245,158,11,0.2);'
    'font-family:Space Mono,monospace;font-size:11px;'
    'letter-spacing:0.12em;color:#f59e0b;cursor:pointer;user-select:none;'
    'display:flex;align-items:center;justify-content:space-between" '
    'title="Click to collapse/expand">'
    '<span>DOWNHOLE DRILLING ANALYTICS &middot; CSV METADATA SUMMARY</span>'
    '<span id="meta-chev" style="font-size:13px;color:#f59e0b;transition:transform .2s">▾</span>'
    '</div>'
    '<div id="meta-body">'
    '<table style="width:100%;border-collapse:collapse">'
    '<thead><tr style="background:#181b22;border-bottom:1px solid #1f232d">'
    '<th style="padding:8px 16px;text-align:left;font-family:Space Mono,monospace;'
    'font-size:10px;letter-spacing:0.1em;color:#8b92a5;width:50%">'
    'HIGH-FREQUENCY FILE (' + str(HF_RATE) + ' Hz)'
    '</th>'
    '<th style="padding:8px 16px;text-align:left;font-family:Space Mono,monospace;'
    'font-size:10px;letter-spacing:0.1em;color:#8b92a5;width:50%">'
    'LOW-FREQUENCY FILE (' + str(LF_RATE) + ' Hz)'
    '</th>'
    '</tr></thead>'
    '<tbody><tr>'
    '<td style="padding:12px 16px;vertical-align:top">' + _hf_rows + '</td>'
    '<td style="padding:12px 16px;vertical-align:top">' + _lf_rows + '</td>'
    '</tr></tbody></table>'
    '<div style="padding:8px 16px;background:#181b22;border-top:1px solid #1f232d;'
    'font-family:Space Mono,monospace;font-size:10px;color:#8b92a5">'
    'HF: ' + _hf_name + ' &nbsp;&middot;&nbsp; ' + str(hf_df.shape[0]) + ' samples'
    ' &nbsp;&nbsp;|&nbsp;&nbsp; '
    'LF: ' + _lf_name + ' &nbsp;&middot;&nbsp; ' + str(lf_df.shape[0]) + ' samples'
    '</div>'
    '</div>'
    '<script>'
    'function toggleMeta(){'
    ' var b=document.getElementById("meta-body");'
    ' var c=document.getElementById("meta-chev");'
    ' if(!b||!c)return;'
    ' var hidden=b.style.display==="none";'
    ' b.style.display=hidden?"":"none";'
    ' c.style.transform=hidden?"rotate(0deg)":"rotate(-90deg)";'
    '}'
    '</script>'
    '</div>'
)

display(HTML(META_BANNER_HTML))
print('Metadata banner ready for HTML report.')


## Cell 5 — Derive Channels & Detect Dysfunctions

In [ ]:
from scipy.signal import welch, butter, filtfilt

SAMPLE_RATE = HF_RATE
WINDOW_S    = 1
WIN         = max(1, SAMPLE_RATE * WINDOW_S)
t           = hf_df['elapsed_time_s'].values
N           = len(t)

# Raw acceleration
ax_raw = hf_df['vib_x_raw_g'].values if 'vib_x_raw_g' in hf_df.columns else np.zeros(N)
ay_raw = hf_df['vib_y_raw_g'].values if 'vib_y_raw_g' in hf_df.columns else np.zeros(N)
az_raw = hf_df['vib_z_raw_g'].values if 'vib_z_raw_g' in hf_df.columns else np.zeros(N)

# Temperature-corrected acceleration
ax = hf_df['vib_x_corr_g'].values if 'vib_x_corr_g' in hf_df.columns else ax_raw.copy()
ay = hf_df['vib_y_corr_g'].values if 'vib_y_corr_g' in hf_df.columns else ay_raw.copy()
az = hf_df['vib_z_corr_g'].values if 'vib_z_corr_g' in hf_df.columns else az_raw.copy()

# RMS — prefer CSV-precomputed; fall back to rolling
def rolling_rms(arr, w):
    return np.sqrt(pd.Series(arr).pow(2).rolling(w, min_periods=1).mean().values)

rms_x = hf_df['vib_x_rms_g'].values if 'vib_x_rms_g' in hf_df.columns else rolling_rms(ax, WIN)
rms_y = hf_df['vib_y_rms_g'].values if 'vib_y_rms_g' in hf_df.columns else rolling_rms(ay, WIN)
rms_z = hf_df['vib_z_rms_g'].values if 'vib_z_rms_g' in hf_df.columns else rolling_rms(az, WIN)

vib_magnitude = (hf_df['vib_magnitude_corr_g'].values
                 if 'vib_magnitude_corr_g' in hf_df.columns
                 else np.sqrt(ax**2 + ay**2 + az**2))

rms_lateral  = np.sqrt(rms_x**2 + rms_y**2)
lateral_g    = np.sqrt(ax**2 + ay**2)
tangential_g = ay
axial_g      = az

# RPM
if 'rpm' in hf_df.columns:
    rpm_inst = np.abs(hf_df['rpm'].values)
    print('RPM from csv rpm column')
elif 'omega_z_dps' in hf_df.columns:
    rpm_inst = np.abs(hf_df['omega_z_dps'].values / 6.0)
    print('RPM derived from omega_z_dps')
else:
    rpm_inst = np.zeros(N)
    print('No RPM source found — RPM = 0')

rpm_series = pd.Series(rpm_inst)
rpm_mean   = rpm_series.rolling(WIN, min_periods=1).mean().values
rpm_std    = rpm_series.rolling(WIN, min_periods=1).std().fillna(0).values
with np.errstate(divide='ignore', invalid='ignore'):
    ssi = np.where(rpm_mean > 1, rpm_std / rpm_mean, 0)

# Shock
SHOCK_THRESHOLD_G = 50
shock_flag = ((np.abs(az) > SHOCK_THRESHOLD_G) | (vib_magnitude > SHOCK_THRESHOLD_G)).astype(int)
shock_1s   = pd.Series(shock_flag).rolling(WIN, min_periods=1).sum().values
shock_max  = pd.Series(np.abs(az)).rolling(WIN, min_periods=1).max().values

# ── LF channels ───────────────────────────────────────────────────────────────
LF_N = len(lf_df)
lf_t = lf_df['elapsed_time_s'].values if 'elapsed_time_s' in lf_df.columns else np.arange(LF_N)/LF_RATE

HAS_INC  = 'inclination_deg' in lf_df.columns
HAS_TEMP = ('temp_external_c' in lf_df.columns or 'temp_internal_c' in lf_df.columns)

if HAS_INC:
    inclination    = lf_df['inclination_deg'].values
    inclination_hf = np.interp(t, lf_t, inclination)
    print('Inclination loaded')
else:
    inclination_hf = np.full(N, np.nan)
    inclination    = np.full(LF_N, np.nan)
    print('No inclination data')

if HAS_TEMP:
    temp_ext    = lf_df['temp_external_c'].values if 'temp_external_c' in lf_df.columns else np.full(LF_N, np.nan)
    temp_int    = lf_df['temp_internal_c'].values if 'temp_internal_c' in lf_df.columns else np.full(LF_N, np.nan)
    temp        = temp_ext
    temp_hf     = np.interp(t, lf_t, temp_ext)
    temp_int_hf = np.interp(t, lf_t, temp_int)
    print('Temperature loaded')
else:
    temp = np.full(LF_N, np.nan)
    temp_hf = temp_int_hf = np.full(N, np.nan)
    print('No temperature data')

# ── Thresholds ────────────────────────────────────────────────────────────────
THRESH = {
    'lateral_rms': {'advisory': 1.0,  'critical': 3.0},
    'axial_rms':   {'advisory': 1.0,  'critical': 2.0},
    'peak_shock':  {'advisory': 20.0, 'critical': 50.0},
    'ssi':         {'advisory': 0.10, 'critical': 0.50},
    'temperature': {'advisory': 140,  'critical': 175},
}

def severity_label(val, metric):
    if not np.isfinite(float(val)): return 'NORMAL'
    th = THRESH[metric]
    return 'CRITICAL' if val >= th['critical'] else 'ADVISORY' if val >= th['advisory'] else 'NORMAL'

def pct(arr): return 100 * arr.sum() / max(len(arr), 1)

stick_slip_flag = (ssi         > THRESH['ssi']['critical']).astype(int)
bit_bounce_flag = ((rms_z      > THRESH['axial_rms']['critical']) & (rpm_mean > 10)).astype(int)
whirl_flag      = (rms_lateral > THRESH['lateral_rms']['critical']).astype(int)
over_temp_flag  = (np.nan_to_num(temp_hf, nan=0) > THRESH['temperature']['advisory']).astype(int)

print(f'\nHF: {N:,} samples @ {SAMPLE_RATE} Hz  |  Duration: {t[-1]:.1f}s')
print(f'LF: {LF_N:,} samples @ {LF_RATE} Hz')
print(f'Stick-Slip: {pct(stick_slip_flag):.1f}%  '
      f'Bit Bounce: {pct(bit_bounce_flag):.1f}%  '
      f'Whirl: {pct(whirl_flag):.1f}%  '
      f'Over-Temp: {pct(over_temp_flag):.1f}%')


## Cell 6 — Shared Plot Helpers

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

MAX_PTS = 4000
DS      = max(1, N // MAX_PTS)
t_ds    = t[::DS]
DS2     = max(1, N // 2000)

ICS_LAYOUT = dict(
    template='plotly_dark',
    paper_bgcolor=ICS['bg_paper'], plot_bgcolor=ICS['bg_plot'],
    font=dict(color=ICS['text'], family='Space Mono, monospace', size=11),
    legend=dict(orientation='h', y=-0.06, font=dict(size=10),
                bgcolor='rgba(0,0,0,0)', bordercolor=ICS['grid']),
    hovermode='x unified',
)
GRID_STYLE = dict(gridcolor=ICS['grid'], zerolinecolor=ICS['grid'])

AMBER_SCALE = [
    [0.000,"#0b0c0e"],[0.125,"#1a1505"],[0.250,"#3d2d00"],
    [0.375,"#7a5200"],[0.500,"#b87a00"],[0.625,"#f59e0b"],
    [0.750,"#fbbf24"],[0.875,"#fde68a"],[1.000,"#ffffff"],
]

def dysfunction_fill_trace(flag, t_full, color, name):
    flag   = flag.astype(bool)
    padded = np.concatenate([[False], flag, [False]])
    diff   = np.diff(padded.astype(int))
    starts = np.where(diff ==  1)[0]
    ends   = np.where(diff == -1)[0]
    if len(starts) == 0: return None
    fc = {'red':   'rgba(239,68,68,0.20)',
          'amber': 'rgba(245,158,11,0.22)',
          'purple':'rgba(139,92,246,0.22)',
          'orange':'rgba(249,115,22,0.20)'}
    xs, ys = [], []
    for s, e in zip(starts, ends):
        x0 = t_full[min(s,     len(t_full)-1)]
        x1 = t_full[min(e-1,   len(t_full)-1)]
        xs += [x0,x0,x1,x1,x0,None]; ys += [0,1,1,0,0,None]
    return go.Scatter(x=xs, y=ys, fill='toself',
        fillcolor=fc.get(color,'rgba(200,200,200,0.15)'),
        line=dict(width=0), mode='lines', name=name,
        showlegend=True, legendgroup=name, hoverinfo='skip')

bb_trace = dysfunction_fill_trace(bit_bounce_flag, t, 'red',    'Bit Bounce')
wh_trace = dysfunction_fill_trace(whirl_flag,      t, 'purple', 'Lateral Whirl')
ss_trace = dysfunction_fill_trace(stick_slip_flag, t, 'amber',  'Stick-Slip')
ot_trace = dysfunction_fill_trace(over_temp_flag,  t, 'orange', 'Over-Temp')

def scale_fill(trace, y_scale):
    if trace is None: return None
    return go.Scatter(
        x=trace.x, y=[v*y_scale if v is not None else None for v in trace.y],
        fill='toself', fillcolor=trace.fillcolor,
        line=dict(width=0), mode='lines', name=trace.name,
        showlegend=True, legendgroup=trace.name, hoverinfo='skip')

def add_dysfunction_shading(fig, traces_rows, y_scale):
    """Add bb/wh/ss shading to specified (trace, row) pairs."""
    for tr, row_n in traces_rows:
        if tr is not None:
            fig.add_trace(scale_fill(tr, y_scale), row=row_n, col=1)

def unavailable_annotation(fig, text, x_paper, y_paper):
    fig.add_annotation(text=f'No data: {text}', x=x_paper, y=y_paper,
        xref='paper', yref='paper', showarrow=False,
        font=dict(color=ICS['red'], size=12, family='Space Mono'),
        bgcolor='rgba(11,12,14,0.8)', borderpad=6)

print(f'Helpers ready | DS={DS} ({len(t_ds):,} pts per trace)')


## Cell 7 — Plot 1: Raw / Corrected / RMS Time-Series Overlay

**Toggle controls** — use the Plotly legend:
- **Single click** on any legend item hides/shows that trace
- **Double-click** isolates that trace (hides all others)
- Groups: X Raw / X Corrected / X RMS | Y Raw / Y Corrected / Y RMS | Z Raw / Z Corrected / Z RMS
- Dysfunction shading bands, RPM/SSI, Temperature, Inclination


In [ ]:
# Plot 1: Time-Series Overlay — 5 rows (X, Y, Z, RPM, Temperature)
# Each vib row: Raw (faded) + Corrected + RMS — toggle via legend
fig = make_subplots(
    rows=5, cols=1, shared_xaxes=True,
    subplot_titles=[
        'X-Axis Lateral — Raw / Corrected / RMS',
        'Y-Axis Lateral — Raw / Corrected / RMS',
        'Z-Axis Axial   — Raw / Corrected / RMS',
        'RPM & Stick-Slip Index',
        'Temperature (°C)',
    ],
    row_heights=[0.22, 0.22, 0.22, 0.18, 0.16],
    vertical_spacing=0.04,
)

rms_max = max(float(rms_lateral.max()), float(rms_z.max()), 0.1)

axes_cfg = [
    (ax_raw, ax, rms_x, 'rgba(59,130,246,0.28)',  ICS['blue'],   ICS['teal'],   1, 'X'),
    (ay_raw, ay, rms_y, 'rgba(20,184,166,0.28)',  ICS['teal'],   '#a78bfa',     2, 'Y'),
    (az_raw, az, rms_z, 'rgba(239,68,68,0.28)',   ICS['red'],    ICS['orange'], 3, 'Z'),
]

for raw_a, corr_a, rms_a, raw_col, corr_col, rms_col, row_n, label in axes_cfg:
    fig.add_trace(go.Scatter(x=t_ds, y=raw_a[::DS],  name=f'{label} Raw',
        line=dict(color=raw_col,  width=0.7), legendgroup=f'{label}_raw'), row=row_n, col=1)
    fig.add_trace(go.Scatter(x=t_ds, y=corr_a[::DS], name=f'{label} Corrected',
        line=dict(color=corr_col, width=1.2), legendgroup=f'{label}_corr'), row=row_n, col=1)
    fig.add_trace(go.Scatter(x=t_ds, y=rms_a[::DS],  name=f'{label} RMS',
        line=dict(color=rms_col,  width=2.0), legendgroup=f'{label}_rms'), row=row_n, col=1)
    for yv, col in [(1.0, ICS['amber']), (3.0, ICS['red'])]:
        fig.add_trace(go.Scatter(x=[t_ds[0],t_ds[-1]], y=[yv,yv], mode='lines',
            line=dict(color=col, dash='dash', width=0.8),
            showlegend=False, hoverinfo='skip'), row=row_n, col=1)

# Dysfunction shading:
#   X row (1) — Lateral Whirl (purple)
#   Y row (2) — Stick-Slip (amber) + Lateral Whirl (purple)
#   Z row (3) — Bit Bounce (red)
for tr, row_n in [(wh_trace,1),(ss_trace,2),(wh_trace,2),(bb_trace,3)]:
    if tr is not None:
        fig.add_trace(scale_fill(tr, rms_max), row=row_n, col=1)

# Row 4: RPM + SSI — Stick-Slip shading
fig.add_trace(go.Scatter(x=t_ds, y=rpm_inst[::DS], name='Inst. RPM',
    line=dict(color=ICS['purple'], width=0.8)), row=4, col=1)
fig.add_trace(go.Scatter(x=t_ds, y=rpm_mean[::DS], name='RPM mean',
    line=dict(color=ICS['amber'], width=1.8)), row=4, col=1)
fig.add_trace(go.Scatter(x=t_ds, y=ssi[::DS]*100, name='SSI x100',
    line=dict(color=ICS['red'], width=1, dash='dot')), row=4, col=1)
if ss_trace is not None:
    fig.add_trace(scale_fill(ss_trace, max(float(rpm_inst.max()), 1.0)), row=4, col=1)

# Row 5: Temperature — use lf_t directly for reliable display
if HAS_TEMP:
    _temp_ext_plot = np.nan_to_num(temp_ext, nan=0.0)
    _temp_int_plot = np.nan_to_num(temp_int, nan=0.0)
    fig.add_trace(go.Scatter(x=lf_t, y=_temp_ext_plot, name='Borehole Temp',
        line=dict(color=ICS['orange'], width=1.5),
        fill='tozeroy', fillcolor='rgba(249,115,22,0.06)'), row=5, col=1)
    if not np.all(temp_int == 0):
        fig.add_trace(go.Scatter(x=lf_t, y=_temp_int_plot, name='Internal Temp',
            line=dict(color='#fb923c', width=1, dash='dot')), row=5, col=1)
    _t_range = [float(lf_t[0]), float(lf_t[-1])]
    for yv, col in [(140, ICS['amber']), (175, ICS['red'])]:
        fig.add_trace(go.Scatter(x=_t_range, y=[yv,yv], mode='lines',
            line=dict(color=col, dash='dot', width=0.8),
            showlegend=False, hoverinfo='skip'), row=5, col=1)
    _tmax = float(np.nanmax(_temp_ext_plot)) if np.any(_temp_ext_plot > 0) else 200.0
    if ot_trace is not None:
        fig.add_trace(scale_fill(ot_trace, _tmax), row=5, col=1)
    fig.update_yaxes(range=[0, max(_tmax * 1.15, 200)], row=5, col=1)
else:
    unavailable_annotation(fig, 'TEMPERATURE', 0.5, 0.08)

fig.update_layout(
    title=f'Time-Series Overlay (Raw / Corrected / RMS) | {SAMPLE_RATE} Hz | '
          f'{N:,} samples | Click legend to toggle traces',
    height=1050, **ICS_LAYOUT)
fig.update_xaxes(title_text='Elapsed Time (s)', row=5, col=1, **GRID_STYLE)
for row_n, lbl in [(1,'g (X)'),(2,'g (Y)'),(3,'g (Z)'),(4,'RPM/SSI'),(5,'°C')]:
    fig.update_yaxes(title_text=lbl, row=row_n, col=1, **GRID_STYLE)

fig.show()
print('Plot 1 rendered. Click legend to toggle. Dysfunction shading: amber=Stick-Slip, purple=Whirl, red=Bit-Bounce.')


## Cell 8 — Plot 2: FFT / Power Spectral Density

In [ ]:
from scipy.signal import welch

NPERSEG = min(2048, N // 4)
def compute_psd(signal, fs=100, nperseg=NPERSEG):
    f, pxx = welch(signal - signal.mean(), fs=fs, nperseg=nperseg)
    return f, pxx

f_x, pxx_x = compute_psd(ax)
f_y, pxx_y = compute_psd(ay)
f_z, pxx_z = compute_psd(az)
f_r, pxx_r = compute_psd(rpm_inst)

mean_rpm = rpm_mean.mean()
f_rot    = mean_rpm / 60.0

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'PSD — X (Lateral)', 'PSD — Y (Lateral)',
    'PSD — Z (Axial / Bit Bounce)', 'PSD — Torsional (Stick-Slip)'
])

psd_data = [
    (f_x, pxx_x, ICS['blue'],   'X lateral', 1, 1),
    (f_y, pxx_y, ICS['teal'],   'Y lateral', 1, 2),
    (f_z, pxx_z, ICS['red'],    'Z axial',   2, 1),
    (f_r, pxx_r, ICS['purple'], 'RPM tors.', 2, 2),
]
for f, pxx, col, name, row, c in psd_data:
    fig.add_trace(go.Scatter(x=f, y=10*np.log10(pxx + 1e-12), name=name,
                  line=dict(color=col, width=1.5)), row=row, col=c)

if f_rot > 0.01:
    for harmonic, label, row, c in [
        (f_rot,   '1× RPM', 2, 1), (2*f_rot, '2× RPM', 2, 1),
        (2*f_rot, 'Fwd Whirl 2×', 1, 1), (3*f_rot, 'Fwd Whirl 3×', 1, 1),
    ]:
        if harmonic < 50:
            fig.add_vline(x=harmonic, line_dash='dash', line_color=ICS['amber'],
                          annotation_text=label, annotation_font_size=9,
                          annotation_font_color=ICS['amber'], row=row, col=c)

    fig.add_vrect(x0=0.1, x1=2.0, fillcolor=ICS['amber'], opacity=0.08,
                  annotation_text='Stick-slip 0.1–2 Hz', annotation_font_size=9,
                  line_width=0, row=2, col=2)

fig.update_layout(
    title='<b>Power Spectral Density — All Channels</b><br>'
          '<sup>Design Guide §3.2 | Welch method</sup>',
    height=700, **ICS_LAYOUT
)
for row, col in [(1,1),(1,2),(2,1),(2,2)]:
    fig.update_xaxes(title_text='Frequency (Hz)', range=[0,20], row=row, col=col, **GRID_STYLE)
    fig.update_yaxes(title_text='dB (g²/Hz)', row=row, col=col, **GRID_STYLE)

fig.show()
print(f'✅  Plot 2: PSD rendered.  Rotary freq: {f_rot:.3f} Hz ({mean_rpm:.0f} RPM)')


## Cell 9 — Plot 3: RPM Histogram

In [ ]:
cmd_rpm = rpm_mean.mean()
low_pct  = (rpm_inst < 0.15 * cmd_rpm).sum() / max(len(rpm_inst), 1)
high_pct = (rpm_inst > 1.8  * cmd_rpm).sum() / max(len(rpm_inst), 1)

if low_pct > 0.10 and high_pct > 0.05:
    diagnosis = 'BIMODAL — Severe Stick-Slip Detected'
    diag_col  = ICS['red']
elif ssi.mean() > 0.10:
    diagnosis = 'Broadened distribution — Mild Stick-Slip'
    diag_col  = ICS['amber']
else:
    diagnosis = 'Single-peak — Healthy drilling'
    diag_col  = ICS['green']

fig = go.Figure()
fig.add_trace(go.Histogram(x=rpm_inst[rpm_inst >= 0], nbinsx=80,
    name='RPM distribution', marker_color=ICS['purple'], opacity=0.75))

for xv, col, lbl in [
    (cmd_rpm,     ICS['amber'], f'Mean RPM = {cmd_rpm:.0f}'),
    (0,           ICS['red'],   'Stick phase (≈0)'),
    (cmd_rpm * 2, ICS['orange'],'Slip phase (2×)'),
]:
    fig.add_vline(x=xv, line_dash='dash', line_color=col,
                  annotation_text=lbl, annotation_font_color=col)

fig.update_layout(
    title=f'<b>RPM Histogram</b><br><sup>Design Guide §3.3 | {diagnosis}</sup>',
    xaxis_title='Instantaneous RPM', yaxis_title='Sample Count',
    height=450, **ICS_LAYOUT
)
fig.add_annotation(text=diagnosis, x=0.5, y=0.92, xref='paper', yref='paper',
                   showarrow=False, font=dict(size=14, color=diag_col),
                   bgcolor='rgba(11,12,14,0.7)', borderpad=5)
fig.update_xaxes(**GRID_STYLE)
fig.update_yaxes(**GRID_STYLE)
fig.show()
print(f'✅  Plot 3: RPM Histogram. {diagnosis}')


## Cell 10 — Plot 4: Vibration Severity Heatmap

In [ ]:
# Plot 4: Vibration Severity Heatmap — time vs frequency bands (no depth)
# Shows lateral RMS, axial RMS, and SSI across time with dysfunction shading
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[
        'Lateral RMS (g) — time heatmap',
        'Axial RMS (g) — time heatmap',
        'SSI — Stick-Slip Index over time',
    ],
    vertical_spacing=0.08)

N_T = 400
tb  = np.linspace(t[0], t[-1], N_T + 1)
tc  = 0.5 * (tb[:-1] + tb[1:])

def time_bin_stat(arr, tb):
    out = np.zeros(len(tb)-1)
    for i in range(len(tb)-1):
        mask = (t >= tb[i]) & (t < tb[i+1])
        if mask.any():
            out[i] = arr[mask].mean()
    return out

lat_binned = time_bin_stat(rms_lateral, tb)
axl_binned = time_bin_stat(rms_z,       tb)
ssi_binned = time_bin_stat(ssi,         tb)

for row_n, data, zmax, colorscale, cbticktxt in [
    (1, lat_binned[np.newaxis,:], 3.0,
     [[0,ICS['bg_paper']],[0.1,'#052e00'],[0.33,ICS['green']],[0.66,ICS['amber']],[1,ICS['red']]],
     ['0g','1g','3g+']),
    (2, axl_binned[np.newaxis,:], 2.0,
     [[0,ICS['bg_paper']],[0.1,'#052e00'],[0.33,ICS['green']],[0.66,ICS['amber']],[1,ICS['red']]],
     ['0g','1g','2g+']),
    (3, ssi_binned[np.newaxis,:], 0.5,
     [[0,ICS['bg_paper']],[0.33,ICS['teal']],[0.66,ICS['amber']],[1,ICS['red']]],
     ['0','0.1','0.5+']),
]:
    fig.add_trace(go.Heatmap(
        x=tc, y=[''], z=data,
        colorscale=colorscale,
        zmin=0, zmax=float(max(data.max(), zmax*0.01)),
        showscale=True,
        colorbar=dict(
            title=cbticktxt[-1], len=0.28,
            y=1.0 - (row_n-1)*0.35, yanchor='top', thickness=12,
            tickfont=dict(size=9, color=ICS['text'])
        )
    ), row=row_n, col=1)

# Add dysfunction shading on each row
rms_lat_max = max(float(rms_lateral.max()), 0.1)
for tr, row_n, scale in [(ss_trace,3,0.5),(bb_trace,2,2.0),(wh_trace,1,3.0)]:
    if tr is not None:
        # For single-row heatmap the y is categorical; use vrect instead
        starts_x = []
        ends_x   = []
        xs = tr.x
        ys = tr.y
        in_band = False
        x0 = None
        for xv, yv in zip(xs, ys):
            if xv is None:
                if in_band and x0 is not None:
                    starts_x.append(x0); ends_x.append(_last_x)
                    in_band = False; x0 = None
            elif yv is not None and yv > 0:
                if not in_band:
                    x0 = xv; in_band = True
                _last_x = xv
        fc_map = {'red':'rgba(239,68,68,0.22)','amber':'rgba(245,158,11,0.25)',
                  'purple':'rgba(139,92,246,0.25)'}
        col_name = [k for k,v in {'red':bb_trace,'amber':ss_trace,'purple':wh_trace}.items()
                    if v is tr]
        fc = fc_map.get(col_name[0] if col_name else 'red', 'rgba(200,200,200,0.2)')
        for x0v, x1v in zip(starts_x, ends_x):
            fig.add_vrect(x0=x0v, x1=x1v, fillcolor=fc, line_width=0,
                          row=row_n, col=1)

fig.update_layout(
    title='Vibration Severity — Time Overview  |  Stick-Slip=amber  Bit-Bounce=red  Whirl=purple',
    height=520, **ICS_LAYOUT)
fig.update_xaxes(title_text='Elapsed Time (s)', row=3, col=1, **GRID_STYLE)
for row_n, lbl in [(1,'Lateral RMS (g)'),(2,'Axial RMS (g)'),(3,'SSI')]:
    fig.update_yaxes(title_text=lbl, row=row_n, col=1, **GRID_STYLE)

fig.show()
print('Plot 4: Vibration Severity Heatmap (time-based) rendered.')


## Cell 11 — Plot 5: Bit Bounce & Shock Event Timeline

In [ ]:
# Plot 5: Bit Bounce & Shock — with Bit Bounce (red) and Stick-Slip (amber) shading
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=[
    'Z-Axis (Axial) Acceleration — Corrected',
    'Axial RMS & Peak Shock (1s windows)',
    'Shock Event Count per Second'
], vertical_spacing=0.08)

# Row 1: Raw Z signal
fig.add_trace(go.Scatter(x=t_ds, y=az[::DS], name='Accel Z (g)',
    line=dict(color=ICS['red'], width=0.8)), row=1, col=1)
for yv in [SHOCK_THRESHOLD_G, -SHOCK_THRESHOLD_G]:
    fig.add_trace(go.Scatter(x=[t_ds[0],t_ds[-1]], y=[yv,yv], mode='lines',
        line=dict(color=ICS['amber'], dash='dash', width=1),
        showlegend=False, hoverinfo='skip'), row=1, col=1)
# Bit Bounce + Stick-Slip shading on raw signal
z_raw_max = max(float(np.abs(az).max()), SHOCK_THRESHOLD_G)
for tr in [bb_trace, ss_trace]:
    if tr is not None:
        fig.add_trace(scale_fill(tr, z_raw_max), row=1, col=1)

# Row 2: Axial RMS + shock max + threshold lines
fig.add_trace(go.Scatter(x=t_ds, y=rms_z[::DS], name='Axial RMS (1s)',
    line=dict(color=ICS['amber'], width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=t_ds, y=shock_max[::DS], name='Peak shock g',
    line=dict(color=ICS['red'], width=1, dash='dot')), row=2, col=1)
for yv, col in [(THRESH['axial_rms']['advisory'], ICS['amber']),
                (THRESH['axial_rms']['critical'],  ICS['red'])]:
    fig.add_trace(go.Scatter(x=[t_ds[0],t_ds[-1]], y=[yv,yv], mode='lines',
        line=dict(color=col, dash='dash', width=1),
        showlegend=False, hoverinfo='skip'), row=2, col=1)
rms_z_max = max(float(rms_z.max()), float(shock_max.max()), 1.0)
for tr in [bb_trace, ss_trace]:
    if tr is not None:
        fig.add_trace(scale_fill(tr, rms_z_max), row=2, col=1)

# Row 3: Shock count bars
fig.add_trace(go.Bar(x=t_ds, y=shock_1s[::DS], name='Shock count/s',
    marker_color=ICS['red'], opacity=0.7), row=3, col=1)

fig.update_layout(
    title='Bit Bounce & Shock Event Analysis  |  red=Bit-Bounce  amber=Stick-Slip',
    height=720, **ICS_LAYOUT)
fig.update_xaxes(title_text='Elapsed Time (s)', row=3, col=1, **GRID_STYLE)
for row_n, lbl in [(1,'g'),(2,'g-RMS'),(3,'Shocks/s')]:
    fig.update_yaxes(title_text=lbl, row=row_n, col=1, **GRID_STYLE)

fig.show()
print('Plot 5: Bit Bounce / Shock rendered with dysfunction shading.')


## Cell 12 — Plot 6: Lateral Whirl Detection

In [ ]:
# Plot 6: Lateral Whirl Detection
# Dysfunction shading: purple=Whirl, red=Bit-Bounce, amber=Stick-Slip
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'X vs Y Lateral — Orbit Plot (Lissajous)',
    'Lateral RMS — X, Y & Total',
    'X & Y Corrected Signals',
    'Lateral / Axial Ratio'
])

# Orbit plot
max_pts = min(5000, N)
fig.add_trace(go.Scatter(x=ax[:max_pts], y=ay[:max_pts], mode='lines',
    name='XY orbit', line=dict(color='rgba(20,184,166,0.5)', width=0.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers',
    marker=dict(color=ICS['red'], size=8, symbol='x'), name='Tool centre'), row=1, col=1)
fig.update_xaxes(title_text='Accel X (g)', row=1, col=1, **GRID_STYLE)
fig.update_yaxes(title_text='Accel Y (g)', row=1, col=1, **GRID_STYLE)

# Lateral RMS with threshold lines + all three dysfunction shadings
for arr, col, nm in [(rms_x, ICS['blue'], 'RMS X'),
                     (rms_y, ICS['teal'], 'RMS Y'),
                     (rms_lateral, ICS['amber'], 'Lateral total')]:
    fig.add_trace(go.Scatter(x=t_ds, y=arr[::DS], name=nm,
        line=dict(color=col, width=1.5)), row=1, col=2)
for yv, col in [(1.0, ICS['amber']), (3.0, ICS['red'])]:
    fig.add_trace(go.Scatter(x=[t_ds[0],t_ds[-1]], y=[yv,yv], mode='lines',
        line=dict(color=col, dash='dash', width=1),
        showlegend=False, hoverinfo='skip'), row=1, col=2)
lat_max = max(float(rms_lateral.max()), 0.1)
for tr in [wh_trace, bb_trace, ss_trace]:
    if tr is not None:
        fig.add_trace(scale_fill(tr, lat_max), row=1, col=2)

# Raw X/Y signals
fig.add_trace(go.Scatter(x=t_ds, y=ax[::DS], name='Accel X',
    line=dict(color=ICS['blue'], width=0.8)), row=2, col=1)
fig.add_trace(go.Scatter(x=t_ds, y=ay[::DS], name='Accel Y',
    line=dict(color=ICS['teal'], width=0.8)), row=2, col=1)
for tr in [wh_trace, ss_trace]:
    if tr is not None:
        raw_max = max(float(np.abs(ax).max()), float(np.abs(ay).max()), 0.1)
        fig.add_trace(scale_fill(tr, raw_max), row=2, col=1)
fig.update_xaxes(title_text='Elapsed Time (s)', row=2, col=1, **GRID_STYLE)

# Lateral/Axial ratio
with np.errstate(divide='ignore', invalid='ignore'):
    lat_axial_ratio = np.where(rms_z > 0.01, rms_lateral/(rms_z+0.01), rms_lateral)
fig.add_trace(go.Scatter(x=t_ds, y=lat_axial_ratio[::DS], name='Lateral/Axial',
    line=dict(color=ICS['purple'], width=1.5)), row=2, col=2)
fig.add_trace(go.Scatter(x=[t_ds[0],t_ds[-1]], y=[2.0,2.0], mode='lines',
    line=dict(color=ICS['amber'], dash='dash', width=1),
    showlegend=False, hoverinfo='skip'), row=2, col=2)
if wh_trace is not None:
    fig.add_trace(scale_fill(wh_trace, max(float(lat_axial_ratio.max()), 2.1)), row=2, col=2)
fig.update_xaxes(title_text='Elapsed Time (s)', row=2, col=2, **GRID_STYLE)

fig.update_layout(
    title='Lateral Whirl Detection  |  purple=Whirl  red=Bit-Bounce  amber=Stick-Slip',
    height=720, **ICS_LAYOUT)

fig.show()
print('Plot 6: Lateral Whirl Detection rendered.')


## Cell 13 — Plot 7: Temperature Gradient Log

In [ ]:
# Plot 7: Temperature vs Time (no depth)
if not HAS_TEMP:
    fig = go.Figure()
    unavailable_annotation(fig, 'TEMPERATURE DATA UNAVAILABLE', 0.5, 0.5)
    fig.update_layout(title='Temperature Log — DATA UNAVAILABLE', height=280, **ICS_LAYOUT)
    fig.show()
    print('Plot 7 skipped: no temperature channel.')
else:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
        subplot_titles=['Borehole & Internal Temperature vs Time',
                        'Temperature Rate of Change (dT/dt)'],
        vertical_spacing=0.10)

    _t_ext_p = np.nan_to_num(temp_ext, nan=0.0)
    _t_int_p = np.nan_to_num(temp_int, nan=0.0)
    _t_range_p = [float(lf_t[0]), float(lf_t[-1])]
    _tmax_p = float(np.nanmax(_t_ext_p)) if np.any(_t_ext_p > 0) else 200.0

    fig.add_trace(go.Scatter(x=lf_t, y=_t_ext_p, name='Borehole Temp (°C)',
        line=dict(color=ICS['orange'], width=2),
        fill='tozeroy', fillcolor='rgba(249,115,22,0.06)'), row=1, col=1)
    if not np.all(_t_int_p == 0):
        fig.add_trace(go.Scatter(x=lf_t, y=_t_int_p, name='Internal Temp (°C)',
            line=dict(color='#fb923c', width=1.2, dash='dot')), row=1, col=1)
    for yv, col, lbl in [(140, ICS['amber'], '140°C Advisory'),
                          (175, ICS['red'],   '175°C Critical')]:
        fig.add_trace(go.Scatter(x=_t_range_p, y=[yv,yv], mode='lines',
            line=dict(color=col, dash='dot', width=1),
            name=lbl, hoverinfo='skip'), row=1, col=1)
    if ot_trace is not None:
        fig.add_trace(scale_fill(ot_trace, _tmax_p), row=1, col=1)
    fig.update_yaxes(range=[0, max(_tmax_p * 1.15, 200)], row=1, col=1)

    # dT/dt — rate of change using LF data directly
    dt_dt = np.gradient(_t_ext_p, lf_t)
    fig.add_trace(go.Scatter(x=lf_t, y=dt_dt, name='dT/dt (°C/s)',
        line=dict(color=ICS['teal'], width=1.2)), row=2, col=1)
    fig.add_trace(go.Scatter(x=_t_range_p, y=[0,0], mode='lines',
        line=dict(color=ICS['muted'], width=0.8), showlegend=False, hoverinfo='skip'), row=2, col=1)

    fig.update_layout(
        title=f'Temperature Log  |  Peak: {_tmax_p:.1f}°C  |  orange shading = Over-Temp',
        height=580, **ICS_LAYOUT)
    fig.update_xaxes(title_text='Elapsed Time (s)', row=2, col=1, **GRID_STYLE)
    fig.update_yaxes(title_text='°C', row=1, col=1, **GRID_STYLE)
    fig.update_yaxes(title_text='°C/s', row=2, col=1, **GRID_STYLE)
    fig.show()
    print(f'Plot 7: Temperature. Peak = {_tmax_p:.1f}°C')


## Cell 14 — Plot 8: Inclination vs Time


In [ ]:
# Plot 8: Inclination vs Time (sensor measures inclination, not depth)
if not HAS_INC:
    fig = go.Figure()
    unavailable_annotation(fig, 'INCLINATION DATA UNAVAILABLE', 0.5, 0.5)
    fig.update_layout(title='Inclination — DATA UNAVAILABLE', height=280, **ICS_LAYOUT)
    fig.show()
    print('Plot 8 skipped: no inclination channel.')
else:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
        subplot_titles=['Inclination vs Time',
                        'Inclination Rate of Change (deg/s)'],
        vertical_spacing=0.10)

    fig.add_trace(go.Scatter(x=lf_t, y=inclination, name='Inclination (°)',
        line=dict(color=ICS['teal'], width=2),
        fill='tozeroy', fillcolor='rgba(20,184,166,0.06)'), row=1, col=1)

    # Rate of change
    inc_rate = np.gradient(inclination, lf_t)
    fig.add_trace(go.Scatter(x=lf_t, y=inc_rate, name='Inc. Rate (°/s)',
        line=dict(color=ICS['purple'], width=1.2)), row=2, col=1)
    fig.add_trace(go.Scatter(x=[lf_t[0],lf_t[-1]], y=[0,0], mode='lines',
        line=dict(color=ICS['muted'], width=0.8), showlegend=False, hoverinfo='skip'), row=2, col=1)

    fig.update_layout(
        title=f'Inclination vs Time  |  Range: {inclination.min():.1f}° — {inclination.max():.1f}°',
        height=520, **ICS_LAYOUT)
    fig.update_xaxes(title_text='Elapsed Time (s)', row=2, col=1, **GRID_STYLE)
    fig.update_yaxes(title_text='°', row=1, col=1, **GRID_STYLE)
    fig.update_yaxes(title_text='°/s', row=2, col=1, **GRID_STYLE)
    fig.show()
    print(f'Plot 8: Inclination vs Time. Range: {inclination.min():.1f}°–{inclination.max():.1f}°')


## Cell 15 — Plot 9: Run Overview (RPM, SSI, Vibration)


In [ ]:
# Plot 9 — Run Overview: RPM, SSI, Vibration RMS with dysfunction shading
# Shows RPM distribution and SSI over full run
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[
        'Instantaneous RPM & Mean RPM',
        'Stick-Slip Index (SSI)',
        'Lateral & Axial RMS',
    ],
    vertical_spacing=0.07)

# Row 1: RPM
fig.add_trace(go.Scatter(x=t_ds, y=rpm_inst[::DS], name='Inst. RPM',
    line=dict(color=ICS['purple'], width=0.7)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_ds, y=rpm_mean[::DS], name='RPM mean',
    line=dict(color=ICS['amber'], width=2)), row=1, col=1)
if ss_trace is not None:
    fig.add_trace(scale_fill(ss_trace, max(float(rpm_inst.max()),1.0)), row=1, col=1)

# Row 2: SSI
fig.add_trace(go.Scatter(x=t_ds, y=ssi[::DS], name='SSI',
    line=dict(color=ICS['amber'], width=1.5),
    fill='tozeroy', fillcolor='rgba(245,158,11,0.08)'), row=2, col=1)
for yv, col, lbl in [(THRESH['ssi']['advisory'], ICS['amber'], 'Advisory 0.10'),
                      (THRESH['ssi']['critical'],  ICS['red'],   'Critical 0.50')]:
    fig.add_trace(go.Scatter(x=[t_ds[0],t_ds[-1]], y=[yv,yv], mode='lines',
        line=dict(color=col, dash='dash', width=1),
        name=lbl, hoverinfo='skip'), row=2, col=1)
if ss_trace is not None:
    fig.add_trace(scale_fill(ss_trace, THRESH['ssi']['critical']*1.1), row=2, col=1)

# Row 3: Lateral + Axial RMS with shading
for arr, col, nm in [(rms_lateral, ICS['amber'], 'Lateral RMS'),
                     (rms_z, ICS['red'], 'Axial RMS')]:
    fig.add_trace(go.Scatter(x=t_ds, y=arr[::DS], name=nm,
        line=dict(color=col, width=1.5)), row=3, col=1)
rms_max3 = max(float(rms_lateral.max()), float(rms_z.max()), 0.1)
for tr in [wh_trace, bb_trace, ss_trace]:
    if tr is not None:
        fig.add_trace(scale_fill(tr, rms_max3), row=3, col=1)

fig.update_layout(
    title='Run Overview — RPM, SSI, Vibration  |  amber=Stick-Slip  purple=Whirl  red=Bit-Bounce',
    height=720, **ICS_LAYOUT)
fig.update_xaxes(title_text='Elapsed Time (s)', row=3, col=1, **GRID_STYLE)
for row_n, lbl in [(1,'RPM'),(2,'SSI'),(3,'g-RMS')]:
    fig.update_yaxes(title_text=lbl, row=row_n, col=1, **GRID_STYLE)

fig.show()
print('Plot 9: Run Overview rendered.')


## Cell 16 — Plot 10: Acceleration Statistics

In [ ]:
from IPython.display import display, HTML

def overall_rms(arr): return float(np.sqrt(np.mean(arr**2)))

accel_stats = {
    'Max Lateral (g)':     float(np.max(np.abs(lateral_g))),
    'RMS Lateral (g)':     overall_rms(lateral_g),
    'Max Tangential (g)':  float(np.max(np.abs(tangential_g))),
    'RMS Tangential (g)':  overall_rms(tangential_g),
    'Max Axial (g)':       float(np.max(np.abs(axial_g))),
    'RMS Axial (g)':       overall_rms(axial_g),
    'Peak Shock (g)':      float(shock_max.max()),
    'Mean RPM':            float(rpm_mean.mean()),
    'Max SSI':             float(ssi.max()),
    'Mean SSI':            float(ssi.mean()),
}

def badge(val, metric):
    if metric not in THRESH: return ''
    s = severity_label(val, metric)
    c = {'CRITICAL':ICS['red'],'ADVISORY':ICS['amber'],'NORMAL':ICS['green']}[s]
    return (f'<span style="background:{c};color:#000;font-weight:700;font-size:10px;'
            f'padding:2px 8px;border-radius:3px">{s}</span>')

badge_map = {
    'Max Lateral (g)': ('lateral_rms', float(np.max(np.abs(lateral_g)))),
    'RMS Lateral (g)': ('lateral_rms', overall_rms(lateral_g)),
    'Max Axial (g)':   ('axial_rms',   float(np.max(np.abs(axial_g)))),
    'RMS Axial (g)':   ('axial_rms',   overall_rms(axial_g)),
    'Peak Shock (g)':  ('peak_shock',  float(shock_max.max())),
    'Max SSI':         ('ssi',         float(ssi.max())),
}

rows_html = ''.join(
    f'<tr style="border-bottom:1px solid {ICS["grid"]}">' +
    f'<td style="padding:8px 16px;color:{ICS["muted"]};font-family:Space Mono,monospace;font-size:11px">{k}</td>' +
    f'<td style="padding:8px 16px;font-weight:700;color:{ICS["amber"]};font-family:Space Mono,monospace;text-align:right">{v:.4f}</td>' +
    f'<td style="padding:8px 14px;text-align:center">{badge(badge_map[k][1], badge_map[k][0]) if k in badge_map else ""}</td>' +
    f'</tr>'
    for k, v in accel_stats.items()
)

display(HTML(f"""
<div style="font-family:Space Mono,monospace;background:{ICS['bg_paper']};
     color:{ICS['text']};padding:20px;border-radius:10px;
     border:1px solid rgba(245,158,11,0.3);max-width:640px">
  <div style="font-size:10px;letter-spacing:0.12em;color:{ICS['muted']};margin-bottom:14px">
    ICS CORE — ACCELERATION STATISTICS
  </div>
  <table style="border-collapse:collapse;width:100%">
    <thead>
      <tr style="background:{ICS['bg_card']};border-bottom:1px solid rgba(245,158,11,0.3)">
        <th style="padding:9px 16px;text-align:left;color:{ICS['amber']};font-size:10px">METRIC</th>
        <th style="padding:9px 16px;color:{ICS['amber']};font-size:10px;text-align:right">VALUE</th>
        <th style="padding:9px 14px;color:{ICS['amber']};font-size:10px">STATUS</th>
      </tr>
    </thead>
    <tbody>{rows_html}</tbody>
  </table>
</div>
"""))
print("Acceleration statistics computed.")


## Cell 17 — Plot 11: STFT Spectrograms


In [ ]:
from scipy.signal import spectrogram as sp_spectrogram

FS       = SAMPLE_RATE
NPERSEG  = 2048
NOVERLAP = int(NPERSEG * 0.875)
MAX_FREQ = 6.0

def stft_amplitude(signal, fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP, max_f=MAX_FREQ):
    f, t_s, Sxx = sp_spectrogram(
        signal.astype(float), fs=fs,
        nperseg=min(nperseg, len(signal) // 4 or nperseg),
        noverlap=min(noverlap, (len(signal) // 4 or nperseg) - 1),
        window='hann', scaling='spectrum'
    )
    amp  = np.sqrt(np.abs(Sxx))
    mask = f <= max_f
    t_hr = t_s + t[0]
    return f[mask], t_hr, amp[mask]

signals = [
    ('axial_g',      az,           'Amplitude'),
    ('lateral_g',    lateral_g,    'Amplitude'),
    ('tangential_g', tangential_g, 'Amplitude'),
    ('rotation_rpm', rpm_inst,     'Amplitude'),
]

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
    subplot_titles=[f'{name} Frequency' for name, _, _ in signals],
    vertical_spacing=0.04)

for row_idx, (name, sig, cb_title) in enumerate(signals, start=1):
    f_hz, t_hr, amp = stft_amplitude(sig)
    vmax = np.percentile(amp, 99) or 1.0
    fig.add_trace(go.Heatmap(
        z=amp, x=t_hr, y=f_hz,
        colorscale=AMBER_SCALE,
        zmin=0, zmax=vmax,
        colorbar=dict(title=dict(text=cb_title, side='right'),
                      len=0.22, y=1.0-(row_idx-1)*0.255, yanchor='top',
                      thickness=12, tickfont=dict(size=9, color=ICS['text'])),
        name=name, showscale=True,
    ), row=row_idx, col=1)
    fig.update_yaxes(title_text=f'{name}<br>Freq (Hz)', title_font=dict(size=10),
                     range=[0, MAX_FREQ], row=row_idx, col=1, **GRID_STYLE)

    # Harmonic lines
    if rpm_mean.mean() > 1:
        f_rot_local = rpm_mean.mean() / 60.0
        for h in [1, 2, 3]:
            f_h = h * f_rot_local
            if f_h < MAX_FREQ:
                fig.add_hline(y=f_h, line_dash='dash', line_color=ICS['amber'],
                              line_width=0.8, row=row_idx, col=1)

fig.update_xaxes(title_text='Elapsed Time (s)', row=4, col=1, **GRID_STYLE)
fig.update_layout(
    title='<b>STFT Spectrograms — Acceleration & Rotation</b><br>'
          f'<sup>window={NPERSEG/FS:.0f}s | overlap={100*NOVERLAP/NPERSEG:.0f}% | 0–{MAX_FREQ}Hz | amber lines = RPM harmonics</sup>',
    height=950, margin=dict(r=130), **ICS_LAYOUT
)
fig.show()
print('✅  Plot 12: Spectrograms rendered.')


## Cell 18 — Inline Summary Dashboard


In [ ]:
from IPython.display import display, HTML

def color_for(val, metric):
    s = severity_label(val, metric)
    return {'CRITICAL':ICS['red'],'ADVISORY':ICS['amber'],'NORMAL':ICS['green']}[s]

# Only metrics from sensors that actually exist on the BHA
param_rows = [
    ('Lateral Vib RMS (peak)', f"{rms_lateral.max():.3f} g",  'lateral_rms', rms_lateral.max()),
    ('Axial Vib RMS (peak)',   f"{rms_z.max():.3f} g",        'axial_rms',   rms_z.max()),
    ('Peak Shock',             f"{shock_max.max():.1f} g",     'peak_shock',  shock_max.max()),
    ('Stick-Slip Index (max)', f"{ssi.max():.3f}",             'ssi',         ssi.max()),
    ('Peak Temperature',       f"{np.nanmax(temp_hf):.1f} °C",'temperature', np.nanmax(temp_hf) if HAS_TEMP else 0),
]

dysfunctions = [
    ('Stick-Slip',    pct(stick_slip_flag), ICS['amber']),
    ('Bit Bounce',    pct(bit_bounce_flag), ICS['red']),
    ('Lateral Whirl', pct(whirl_flag),      ICS['purple']),
    ('Over-Temp',     pct(over_temp_flag),  ICS['orange']),
]

table_rows_html = ''.join(
    f'<tr style="border-bottom:1px solid {ICS["bg_card"]}">' +
    f'<td style="padding:8px 16px;color:{ICS["muted"]}">{n}</td>' +
    f'<td style="padding:8px 16px;text-align:center;font-family:Space Mono,monospace;color:{ICS["text"]}">{v}</td>' +
    f'<td style="padding:6px 14px;text-align:center">' +
    f'<span style="background:{color_for(raw, m)};color:#000;font-weight:700;font-size:10px;' +
    f'padding:3px 9px;border-radius:4px">{severity_label(raw, m)}</span></td></tr>'
    for n, v, m, raw in param_rows
)

dysfunc_html = ''.join(
    f'<div style="margin:7px 0;display:flex;align-items:center;gap:10px">' +
    f'<span style="min-width:130px;color:{ICS["muted"]};font-size:11px">{nm}</span>' +
    f'<div style="flex:1;background:{ICS["bg_card"]};border-radius:4px;height:18px">' +
    f'<div style="width:{max(p,0.5):.1f}%;height:100%;background:{col};border-radius:4px;opacity:0.85"></div></div>' +
    f'<span style="min-width:45px;text-align:right;color:{ICS["text"]};font-size:11px;font-family:Space Mono,monospace">{p:.1f}%</span>' +
    f'</div>'
    for nm, p, col in dysfunctions
)

display(HTML(f"""
<div style="font-family:DM Sans,sans-serif;background:{ICS['bg_paper']};color:{ICS['text']};
     padding:24px;border-radius:12px;border:1px solid rgba(245,158,11,0.3);max-width:760px">
  <div style="display:flex;align-items:center;gap:14px;margin-bottom:20px;
       padding-bottom:14px;border-bottom:1px solid {ICS['bg_card']}">
    <div>
      <div style="font-family:Space Mono,monospace;font-size:10px;letter-spacing:0.15em;
           color:{ICS['muted']}">ICS CORE SYSTEM &middot; DRILLING RUN SUMMARY</div>
      <div style="font-family:Space Mono,monospace;font-size:17px;font-weight:700;
           color:{ICS['amber']};margin-top:2px">ZerdaLabs Downhole Analytics</div>
    </div>
  </div>
  <div style="font-family:Space Mono,monospace;font-size:11px;color:{ICS['muted']};margin-bottom:16px">
    Duration: {t[-1]:.0f}s &nbsp;&middot;&nbsp; {N:,} samples &nbsp;&middot;&nbsp;
    {SAMPLE_RATE} Hz &nbsp;&middot;&nbsp; Mean RPM: {rpm_mean.mean():.0f}
  </div>
  <h3 style="font-family:Space Mono,monospace;font-size:10px;letter-spacing:0.12em;
       color:{ICS['muted']};margin:0 0 8px">SEVERITY PARAMETERS</h3>
  <table style="border-collapse:collapse;width:100%;background:{ICS['bg_panel']};
       border-radius:8px;overflow:hidden;margin-bottom:20px">
    <thead><tr style="background:{ICS['bg_card']}">
      <th style="padding:9px 16px;text-align:left;font-family:Space Mono,monospace;
           font-size:9px;letter-spacing:0.1em;color:{ICS['amber']}">PARAMETER</th>
      <th style="padding:9px 16px;font-family:Space Mono,monospace;font-size:9px;
           letter-spacing:0.1em;color:{ICS['amber']}">PEAK VALUE</th>
      <th style="padding:9px 16px;font-family:Space Mono,monospace;font-size:9px;
           letter-spacing:0.1em;color:{ICS['amber']}">STATUS</th>
    </tr></thead>
    <tbody>{table_rows_html}</tbody>
  </table>
  <h3 style="font-family:Space Mono,monospace;font-size:10px;letter-spacing:0.12em;
       color:{ICS['muted']};margin:0 0 8px">DYSFUNCTION OCCURRENCE (% OF RUN)</h3>
  <div style="background:{ICS['bg_panel']};padding:14px;border-radius:8px">
    {dysfunc_html}
  </div>
  <div style="font-family:Space Mono,monospace;font-size:10px;color:{ICS['muted']};margin-top:14px">
    {"No temperature data" if not HAS_TEMP else f"Borehole Temp: {np.nanmax(temp_hf):.1f}°C peak"}
    &nbsp;&middot;&nbsp;
    {"No inclination data" if not HAS_INC else f"Inclination: {inclination.min():.1f}°–{inclination.max():.1f}°"}
  </div>
</div>
"""))
print("Summary dashboard rendered.")


## Cell 19 — HTML Report Generator

Generates a self-contained HTML report file including:
- CSV metadata banner at the top
- KPI strip
- Aggregated stats table (max / mean / min / std dev) for all 1 Hz and HF channels
- All 9 interactive Plotly figures (each collapsible via HIDE/SHOW button)
- Dysfunction occurrence bars


In [ ]:
import plotly.io as pio
from datetime import datetime

print('Building HTML report...')

LOGO_DATA_URI = 'data:image/png;base64,/9j/4AAQSkZJRgABAQAAAQABAAD/4gHYSUNDX1BST0ZJTEUAAQEAAAHIAAAAAAQwAABtbnRyUkdCIFhZWiAH4AABAAEAAAAAAABhY3NwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAQAA9tYAAQAAAADTLQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAlkZXNjAAAA8AAAACRyWFlaAAABFAAAABRnWFlaAAABKAAAABRiWFlaAAABPAAAABR3dHB0AAABUAAAABRyVFJDAAABZAAAAChnVFJDAAABZAAAAChiVFJDAAABZAAAAChjcHJ0AAABjAAAADxtbHVjAAAAAAAAAAEAAAAMZW5VUwAAAAgAAAAcAHMAUgBHAEJYWVogAAAAAAAAb6IAADj1AAADkFhZWiAAAAAAAABimQAAt4UAABjaWFlaIAAAAAAAACSgAAAPhAAAts9YWVogAAAAAAAA9tYAAQAAAADTLXBhcmEAAAAAAAQAAAACZmYAAPKnAAANWQAAE9AAAApbAAAAAAAAAABtbHVjAAAAAAAAAAEAAAAMZW5VUwAAACAAAAAcAEcAbwBvAGcAbABlACAASQBuAGMALgAgADIAMAAxADb/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCAG/AZwDASIAAhEBAxEB/8QAHQABAAICAwEBAAAAAAAAAAAAAAcIBQYCAwQBCf/EAEoQAAICAQIDBQMIBwUGBAcAAAABAgMEBREGByESMUFRYRMicQgUMoGRobHRFSNCUmKywRYzcoKSJCZDU+HwNDZzohclJ0RUdNL/xAAbAQEAAgMBAQAAAAAAAAAAAAAABQYBAwQCB//EADgRAAEDAwEFBgMIAgIDAAAAAAABAgMEBREhBhIxQVETImFxscGBkaEUIzI0QtHh8DNSFWIWcvH/2gAMAwEAAhEDEQA/AKZAAAAAAAAAAAAAAAAAAABde4AA3TTuXmq6jy4nxlhTjdXXlvHnjxXv9FHqvt+40s1slZIqo1c40U9OYrcKvMAA2HkAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAz/CHCWs8U22w0nGdvsnBTfgnJ7L/v0PvMLh+PC3FeVoSud0sVQjZPzm4pv72au2YsnZ51PW47d3uRr4ANp5AAAAAAAAAAAAAAAAAAAAAAAAAAAAAABu3I3Ao1HmjouNk1wtpdrlOE1upJRb2+40k2/k7qVek8xNLzbGlGM5R3fg5RaX4nPVo5YHo3jhfQ2Q47RueGSxfLvDnkcvdfw449eIlrUrI1R7lB9hru80VIuXZtnF+Emi5nKTI03F4KlGuyV1mSlfOcurlKScUvqcGvqKpcT6VRh6RiZ8FJXZOXkRnv3JRcez/MyAssq/a6hqpoqpj5a+h31rPuo1Tovqa6D2Y2n5F1MLlBqqc1BWv6EX/E/A6MvHuxb5U3wcZxez/wCnmWZHIq4I3CnUADJgAAAAAAAAAAAAAAAAAAAAAAAAAHv0HR9T13Uq9O0nDty8mx7RhBb/AFvyRhVRqZUImTwAkXjPlHr3C3DK1rOy8KxqcYWY9VnanFv8WunRHm5McDV8accLQtReTi0xonbbKC2lHZdN9/N7HItfT9i6dHZa3OceBuSnkV6MxqpMXyU9Cycbh16v2E4X3ytaa74whKMdv8zf2ER898eOLxhOF6i9QucsnKalu4ub3jB+qRc3hPQcDhvh7E0fBhGNONSq99tu15v63uykfO7Pq1Lmpr+RT/dxynWv8qUX96ZVrDcFuFymlandx76f3wJSvp/s9Oxi8TTAAXUhQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAc6ZyrthZB7Si00zgc6IOy+EE9nKSQUFu+Vmm/o7l6tVvmrMquqDlGLe0Ybe1S289rX9ppfM3hrDfDEtMvhOqNGo02OyFe8oxkpQk0vhBMmbhbTfmnBGNgXKE5/NYwta7ptQUd/sSIr+UxkZeDgQ1PCtjCMp4tkHv3uDs8PH6UT59bq10twcxq8VX6f1Sy1VMkdMjl6Gv8EaVVwrJcNcSafTl6dq82sXU4VuVT37oz8t/tT+A475Wy0ujb2NmXoli7deRU1O7C38f46zO8neP9E1zBWk6lXGN0pKV2NNe45f8AMr8n5r/tTlPTsW/AVVNjjX31yXXsvx+p+K7jdcbpNQVOXpjPHovinRTRTUjJ49FKH8X8M6jwznxx82KnTau3RkQ6wtj5p/0MIXp4y5e6JxDpEsHJxIOKj0jHolLwlHyfwKnc0OXWq8GZ05uFl+nSltC7brD+GXk/XxJ6032CvTdzh3Q4Kugkp9VTQ0cAE6cAAAAAAAAAAAAAAAAAAAB9hGU5qEIuUpPZJLq2T9yd5A5eqwx9Z4tUsXFbU4YbW07I/wAXl8DirrhT0MfaTuwnqboIJJ3brEyRVy14I1TjjWngafFqFce3bZt0ivw3LXcFcrNO4P4elHEhbZkyrbyFCW07nt9HteC+Bv3DXDui8OYSw9F07HwqV3quCW/x8z7xNrmDoGmTzc1yl4V1QW87ZeEYrxZ83uW09RXzJHTphvTmpYqe2RwMV0i5U1LUdH4d0nB/tLxBVTB40N6q8qz9TS9u5LxfrtuaHyI1HA4l5r8V8S4koOt0V1V9mrsLZ97S8vdI2508VajrOV874mujCak1haLVZuqI/v3bftehn/kl16jivUdRrhTGjKyKsd2Se78W0or6upMSW2WC2SySu77kx4ImeCdfP1OWOoa6qY1qaJr/ACWM4qz3gcPZ2VCUFbCiTr7Utk5bdFufnpmXW5OXdkXyc7bZynOTfVtvdl0/lDZVuNy/yZY9Eb7tpOMXPbsrsveW3jsupSc6diqdGUr5f9l9DxfH5la3ogABdCEAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAByqk42Rku9NM4gAvvwffLI4R0i2ffZhUyf1wRGfymMfSreFsajMudVqjN4sUm+1KMd0vgbpyoy/nXLjQbm238yhH/Stv6Guc+9NlnaPpmRBtSpzIxb27lLofKbcvZXfjhN53uXmrZvUGfBPYqRiZN+JkQyMa2dVsHvGcXs0Wa+TPzOnqcLOG+IM6HzqC3xZ2PaVvXrHfzKx3La6a8pM54eTdh5VWVjWSruqkpwlF7NNdx9FuVuiuECxP58F6FPpal9NIj2n6Kt7/WYzW9I07V8S3F1DFrvptj2Zxkt015EacmOb+ncU4VOl6rNYusVxUXu/du9V6+hK0mfIqmkqLbPuO0VC7QSxVUeU1RSpnOfk1m8LO3WdCjPL0nftTrS3nQv6x9SHz9Db4wtrlXZCM4SW0otbporzzs5Lxav4g4RoSl1nkYUe5+bh+ReLFtOkuIKvR3JevmQVxsysRZIOHNP2K8A5TjKE5QnFxlF7NNbNM4l2K6AAAAAAAAAAAADIcP6Lqev6pTpuk4luTk3SUYxhHfb1b8Eblyh5Xavx3nxtalh6TXL9bkyj9L+GPmy3vBfCPD/CWm14WjYFVKivescd5zfm33lbvO0cFu7je8/p08yUobXJU95dGmh8neSOkcKRq1TXPZ6jqzSezW9dL/hXi/UmSL2Wx0xkL8inGonkZFsK6q05SnJ7KKXi2fLa6tqK+Xfmdlf7wQs8NPHTs3Wpg7b766KnZY0ku5eZXHnzzfpwc+zSuH54+Xnwi4yyl70cbfvUPBy9fAw3PjnVPUL79B4TvccdbwvzYPZz9IeS9SAZNyk5SbbfVt+JfdnNmuxRKipTVeCFfuVx7Rezj4dTsyr7srInkZFs7bbJOU5ye7k/MtX8k3ScjG4Esz8iNfssnJlOjbv6Lsvf6yp5dzkNj/M+U+g1tbOdDsf+aTZ37XzrFQo1v6lRPc82SNH1OV5Iad8rPUcjC4d0+FcZezulZCbT2747bfDqVULb/Kp02OZy9efK3b5nZGUYfvNtL+pUg2bJK1ba3HHK5+Z4vKKlUuQACzESAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAXM5H29vlVoUvKiS+yckdXOh5K4YxrqJbRqzK52fDfp9+x4vk+5cL+Velwi+tLsrl8e23/VGY5ouuXBmbGxrbZNLbfdp7r70fKsLHd1yn61+qn0DCPtyKn+qehTbVYyjqeVGf0ldJS+O55jJcU1+x4k1Gr93JmvvZjT6mxctRSguTCqdmNfdjXwvx7Z1WwalGcXs0yznI3nHXrNdPD/E16r1FbRoyZdFd5J/xfiVfOVc512Rsrk4zi94yT2aZw3K2w3CLs5E15LzQ6KSrkpX7zPkfoQ5JrdPdHCb6dSsnKbnbmaVbDTeKp2ZeHLaMMjvnV8fNFj9Pz8TUsKvOwcivIx7Y9qFkHumj5bcrPPbno2RMt5LyLzQVsVYzLePTmQn8oPlhRlYlvE3D2FJZkX2sqiqPSyPjJLzRXFpptNbNd6L+WNSTTSafgyE+cvKGjVY263wzRGnO3crsddI3eq8pfiWjZ/aBGtSmqV8lX0Uh7vZXOVZ4E80/YrcDty8e/EybMbJpnTdW+zOE1s4s6i8oudUKoqYAAMgAAAEj8j+XWTxnr1eTmY81ouNPe+zuVj/AHE/xPfye5SZvFVleq6yp4mkJppNbTv+HkvUtNoem6fo+nVafpmLXjY1S2hXBbIqd+2iZStWGBcv69P5J62Wh86pLJo31/gyGl4eJpuFVh4OPXj49UVGFcIpJJHtjLx3PJGZrXFHMThXhzDy7s7VKXbivsyprknZKW3SKXmfN46eapfhiK531LRK6OFveVEQ2nVtTwdI067UNRya8bGpi5TsseySKnc7+cufxbdfo2h2WYuiJ9mTXSeR6vyXoa9zZ5o61x5lexm3h6XXLerFhLo/Jy82R+fRbBsyyjxNUJl/JOSfyVK43RZ1VkX4fUAAt5DHKCcpxiu9vYvpwLjywuDNGxJR7MqsGqMl69hblF9Ax5ZWt4ONFbytyIRS+MkX5oXs6YVrooxSX1FG21k7kUfiq+3uWXZ2PLnu8kIm+VbnVU8B0407NpX5CSh4y26lUSyHytcvGhhaZjW1uds1N1+Ue7qVvJjZePctzNOOfUj707NW74AAFhIoAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAtB8mKT/APhxYm9/9us2X+WJt/Mqu6/g/NhjzjCxpbN92zaT+7c0j5MVifL/ACK0+sc6e/1xiSDxXXG/hzPrmotOiT2l3dFufM7h3buq/wDZPY+hULN+1on/AFX3KfcZw7HE2a/35Kz/AFRT/qYcznG9Tr1vt7Ps201yi299/dSf3pmDPpEK5jb5FBlTD1AANhrBvvKvmVqvBebGmUpZWlzl+tx5P6PrHyZoQNNRTxVEaxyplFNsMz4Xo9i4VC8vDfEGmcR6RVqml5EbqLV5+9F+Ka8Ge+cimXL7jbWODdUjk4Frnjyf67Hk/csX5+pabgrjLR+LtLjl6belYl+tok/frfk15ep83u9ikoXK9urOvTzL5artHWJuu0f06+RhuZXLXReMIPI2+Z6lFe7kVr6XpJeJWjjLhXV+FdTlhanjyit/1d0V7li9H/QuZOXevtMPxNoum8QaVbp2p48bqbFt1XWL80/BnRaL7LSKkcq7zPqnkYudijqkV8aYf9F8ylwNr5l8G5XB2trFsn7bEu3ljW+MorvT9VujVD6FDMyZiSMXKKUOaF8L1jemFQJbvZE48oeTrylja9xQtqJJWU4fjLyc/T0NR5GcLYvEHFMMnUdpYeI1P2b/AOLPvS+HTqWprklFRS2S7l5FX2iu8lNiCFcKvFSx2G0tqPv5UyicEPbjRrpqhVTBQrgtoxitkl5Ho9rGEXOUlGMVu233GPlfXTVK62cYVwTcpSeySXiV150c3MjVrMjQOHLnVp69y7Ii9pXeaT8I/iU63Wma4y7reHNSw3Ctioo8u48kNi50c6o1wu0DhK5+13cL82PcvNQ/Mr1fdbfZK26ydk5PeUpPdtnWD6db7dBQR9nEnmvNSg1VXJVP33qAAd5ygAAGz8qaHk8xtBqS33za216KW5d3tFPvk74kcrmnp0pJtURnb0W/VRe33lu3PZN+R872wfv1MbOieqlx2cjVIHv6r6EAfK2VUrdLmk3Yt4yfa6bd+239SACbvlJZs7sPCryK/wBdZlWTrm47N1pbJPr5tkIlusbOzoY29Cu3N29VPUAAljgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABkeHsCjU9UqwLslYzufYhZJe6peG/oYVURMqZRFVcITT8lzUqFgarpc74q72sLa62+sls02vsRNGQlbj2VtJqcWtn47lUFwpxvwvrbyMLCylfjL2kb8ddqLj5p+PwJy5W8wsXivC+aZnYxtXpW1tL6Kzb9qP8AVeBRb5b1dKtXCu8i4zjkXmx1yNiSlmTdcmcZ5kJc3MOGHn6bUq5VzWPOMot7pdm2aNHJU5+TxJPTI0WOVkbstT3WzW9m/wDUisuFA9XwNVfH1KhXxpHO5qeHoAAdZyAAAAyXDet6jw/qtWo6ZkTpurfg+kl4p+aMaDy5qPRWuTKKemuVqo5q6ln+DebvDmuQqx86z9G5sopSjb/duXpL8zbZ69pMbZ12ahjVOK3TnbFKS8099mUzO6zKybKo1TvslCPdFy3SK3NsxTufvRuVpZKfaeojbiRqOJT5+8U6BrtmPg6dKd+XhWtSuj/d9lrqk/Hrt9hEwBO0dKylhSJnBCDrKt9XMsz+Kkv/ACZtTwcbXc7Trov53lVqVEvDaKbkv6/UWAvzMfEx5X5V9dFUVvKc5JJfWylWmahm6ZmwzdPybMbIhv2bK3s1utmerU+Idc1On2OfqmXkV779idja3IS42D7bU9rv4TmTVvvyUdL2W5lUzj+SR+dXNC7W77NC0K+VemVvs23QezyH5f4fxIkAJukpIqSJIo0wifUg6qqkqpFkkXVQADpOcAAAAAAl75K7qXHOY5v3/mUlD/VHf7izLl0exWv5K+PN8VajlqtuFeL2XP8AdbktvwZYq9v2E0pdl9lrfy6HzjahN64J5IXzZ5MUWcc19iuPyl8ym3VNJw4Xwssors7cYvfs7tbdfqZEBvPOmyf9qKsayuXaox4x9rLvt8e19rZoxfKFiMgaiFLqnb0zlAAOs5wAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAduHc8bKqvit3XJSX1HUAqZCLjUtRyv4pwOJOH61VkOWXRHs312fTj6+q9TXOZ/L66zI/tNwp2sXVKX2511Ps+09V5S/Egrh/WdQ0LU6tR03IlTdW/B9JLxTXiizHLbjjA4u05bOFGfWv11Df3x80U6vo5rbMtTBq1eKefJfAu9vroLpAlLUaPTgvuniV/494oy+I/mEdQxfY5uHCVd8ttnOTfe14M1csNzf5b1a3RbrWjVRr1KCcrK0tlev/wCivdkJ12Srsi4Ti9pRa2afkWG21cNTCixaY4p0K1dKKelnVJtc8F6nEAEgRoAAAAAAAAAAAAAAAAAAAAAAAAAABO/yVVBVa5N7qXaq+z3ibM+5V4V09t+zBvbfbch35L1fY0DVbt4+/kRj69F/1JeyIwuonVZs4TTT3W/RnzW+YdcnfD2PotlYqW5uPH3Kvc8suOTxdTDtKVlGJCFjT3W/V7J/Bo0E3XndGMOZOp1wSjCHYUUvBdhGlH0KkcjoGOTmiFCqmqyd7V5KqAAHQc4AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAPZo2p5ukajVn6ffOi+p7xlF/ceMGFRHJhTLXK1cpxLT8suN8Pi7SkpONWpVR2vp8/wCKPoaBz84JlG3+0uk4vuNP57GC8fCe34kTaDq2domqU6jp18qb6pbpp9/o/QsLy95hYHFVcsHUVVTkWR7Ps5PpJ7dY9e8q89DLbZ/tNMmW80LZT18Vzp/stUuHcl8StYNu5scOPhzi/Jpqo9lh3v2uPt3dl96+pmollikbKxHt4KVaWJ0T1Y7imgABsNYAAAAAAAAAAAAAAAAAAAAAAABYH5Mj24Y1LzeUv5TZ+POY+icKr2M5PLzmulFTXu/4n4ECcI8X65o2g5miaLS/aZdik7YRbnHpt0N54E4D0iOH/aDi/Itutb7TptXR+vnIqdZa4/tL6mo1bnRE4qW+iukn2VlNTph2NXLwQjDivWcjiTiLK1e6mMLciSfYhu0kkkvwMQTNrGi5ep5s6uF+G6sbCyY7e1nT2JL6/AijiDS8nRtVu0/LUFdVLaShLdFhpZWOajGJjCcCt1cMjXK965yvHqeAAHWcgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAOdNtlNsbaZyrnF7xlF7NM4AA9mq6pqGq3+31HLtybF3SslvseMAwiIiYQyqqq5UAAyYAAAAAAAAAAAAAAAAAAAB79C0jP1vUq9P03HldfY+iXcl5t+CMOcjUyplrVcuE4nhSbaSTbfckSLwFys1XXVDM1NWafhNprtR2nNeifcSfy95Y6Vw7TVmZtUc3U9k3KxbxrflFf1N/UI7p7bbLYq9ftA1Msg+f7Ftt2zjnYfUfL91Na0ThDS9Ao9lomDRVc4KMsixdqTX9TIVaLhY/wDtWY5ZdsFv27Oqiu/ou5Hn4w4u0XhbFd2pZKVrXuUw6zl9RAfHnMzW+JJSx6JywMDfpVXLrJfxPxI+jp62vXOVRq8V5qSVdU0FuTGEV6cE5J5m+c0uY9GH29L0fKTmo7SlQ/u38PqIOy8izKyJX2ycpyfVt7nS2292C30lHHSs3WIUqsrZat+89QADqOQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAH1Jt7JbsyHD+i6lr2oQwdMxZ33S79l0ivNvwRP/LvlZpegRhmaooZ+odJJyj7lT9F/Uj665Q0TcvXXoSVvtU9e7Eaac15EY8D8q9b1+FeXmf/AC/Ck095r35r0ROvB3COjcLYfsdNx0rJL9ZdPrOfxZn4pRilFJJdyXganx1x9ovClThfasjMa9zHre7+vyRTai4Vlzf2bE0XknuXimttFamdq9dU5r7G122QqrlZZOMIR6ylJ7JES8wub2NgTswOHIwyb1vGWQ/oRfp5kc8ecxtb4ok6e28PC8KKpP3v8T8TSiat2zzY8PqNV6ciCue0r5Mx0uideZ69W1LO1XNnmahk2ZF8++U3ueQAs7Wo1MJwKm5yuXKgAGTAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAANp4A4K1PizOUaYujDg/1uRJe6vRebPbyo4Oo4p1Ocsy5xxsdx7VUPp2N+Houj3ZZLRNNx9LwK8THqrqrgklXWtox2/77yDu13bSIrGfi9Cfs9ldWuR7/AMPqePhHhjSuGNOjh6bQovZe0tfWdj82zJ5+Xi4GLZlZl9dNFa3lOctkjHcW8S6Xwzpks/Ur1Fd0K11lY/JIrbzA451XizNk7pyowYv9VjRfRLzfmys0NtqLlJ2ki6c19kLXcLpT2qNIok73JE9zeOYXOG2/2mn8Mp1V9YyypfSf+FeHxIgysi/KyJ5GTbO22b3lOb3bZ1AvFJRQ0jN2JMepQayunrH78q59EAAOo4wAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAF1eyAAPs4yhJxnFxkns01s0fAAAAAAAAAAAAADtxqL8m1VY9Nl1kuijCLk39SM9pvAvFuoxcsbQstpd7nHsfzbG18jeK9J0XPnp+qUYtML3vHLlDeSl02i34IsLj3U30xuoshbXJbxnCSafwaK9dLxNRSbiR6dVLLabHDXRb6ya80TkVclyz42it/0Ha/hOP5mB1vQtY0W906pp+RiyX78ej+vuLi7Ebc6OK9I0/Q8jSXbTfm5FUowrjFT7D6fS/d8fU5qG/z1MyR9nx6HVcdnaelgdL2i6dSJ+VHGdPCOpXyycX2tGSkrLIfTglv3eHUknUudmiw0v2mFhZFubJPaufSMX4NvxIABM1Vqpqp+/ImpB0l3qaSNY4lwhluKOINT4j1KedqeRKybfux392C8kvAxIB3ta1iI1qYQjnvc9yucuVUAA9HkAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAGY4O0fI13iTD03G29pZPfd+CXV/cjDkw/J14bjdm2cSW29n2EpU0w2+k2ur+xnLWz/Z6d8nRDroKf7TUMj6qR5zCx/mvG2rUdnsqOTLZfHqYEmL5QvDeFjZNeuY0XXdf/frwm+iTIdMUNQ2ogbI0zX0zqaodG7qAAdZxgAAAAAA9GFh5WZZ7PFx7LXut+xFvb4m4cveXepcS2LJylPC05bN2yWzn17opk28NcNYGjYcaNEorjS23PIthvJbLZ7b97fXv6I4Kq4RQd3OVJGktstRrjCFac/SsrEnbH2crY07e0shFuEW/U2blvx/qfCudVVZbZkaXKX63Hk99k/GPkzfudVWHiaJVg0ZUcahbydSS3tl5vxfeQYZiVlbB326KeZUkoJ/u3aoTJzC5wPLxJYHDHtqO30nlTW0ttuqivD4kPXWWXWSstnKc5Pdyk92zgDZS0cNIzciTBrrK6asfvyrn0QAA6jkAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABY35Ode3AVk2998ybXp0iVyLJfJ7hKPL6LfdLJsa+78iE2gdu0S+aE9s2zer2+Sndz8x4Xcu8u2UU5VWVyi9u73kv6lZy0/OHF+ecAahV2+y9ovf4SRVg07NOzSq3ovsh0bVMRKxHJzT3UAAsJWQAbBwPwnqfFeqLDwYdipdbb5L3K1+foeHvbG1XOXCIe443SORrUyqmAjGUpKMU233JLvJT5bcCVvHWq6tjWZGQvepworrt5y37t/U33hLlfo2h5FWVVZZl5UItO6zZxi/4Ym5P5ppePZ81qinHeVktui82/wAiv1l7Yvcg1VSyUVhen3k64RDlouHZi0drIhXVGMfcqj9GtGice8xKcJW4ekShbOvdTt3/AFMH6v8Aafojwcy+MVThyoyMi7GVie2PW9rbvJy/di/LvIS1XU79Qs99Rqpi/cph0jExRW50z+2n+CGa+5tgZ2EHxXmenXddy9VussybJXWTfvWTe7+C8kYgAsTWo1MIVlzlcuVAAPR5AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABZvkOv/pvh+tln8zKyFneRS7PLfA9ZTf8A7mV/aT8njxQsmy3574KbHxpRXk8KanVZFSTxpvZ+aTaZT8uLxT/5a1L/APWs/lZTo59l1+5enj7HRtan30a+AAJL5Rcu8jXcuvV9WplVpdb7UYyWzvfkvQsNTUx00aySLhEK3S0slVIkcaZVTq5XctcriOyGoamp4+mp7pbbSt+HkiwelaTgaThxw9OxqseiK2cYR239T0Y1FePXCqmEa64LaMYrZJHzMyaMPHnfkWwqqgnKUpPZJFBrrnNXSIjeHJD6Lb7VBQR5dqvNRcn2VCMlVWu+Xdt8CJOZvM3H0x2aPoKrtyItxsu37UYP+rMBzR5o3apKzS9CnKrFW8Z5C6Sn/h8kRU222292yxWqz9kiSTcehWLve1mVY4V06/sd2dl5OblTycu6d11j3lOb3bZ0AFjRMFaVc6qAADAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAB9jt2lut1v1Lz6Fr3AOp8l+GKOFtNeNlVURhavZ9lwlFbWbv9reW7+souWZ5LtR5caZstm1Nv49uRB7QSKykXCcVwT+zcSSVqZXgmTb9dlF6Lm7+NE/wZTh9WW64klKXD+oRju5PHml/pZC/IXg/T9dzMrVdTrV1WHKMa6ZL3ZSe73fw2IuxVDKSlllfwRUJjaGlfV1cMLOKop7OT3LP9IKrXeIKWsXftY+NJf3n8UvQnWmuuqqFdUIwhFbKMVsl6HKMYxioxSil0SXgjw6/q+DoemW6jqN8aqK1u2+9vyXqQVZWzXCZM8+CE9RUMFtgXHLip2avqOFpWDbnZ98KMepdqUpP/AL3ZW/mdzDzeKcmeJiOeNpcZe7Xv71nrL8jy8y+Os7i7P7KcqNOqf6mhPv8A4pepppb7RZ0pU7SXV/oUy9Xt1W7sotGJ9QACeK6AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACynJ2W3LrTF6WfzyK1ljuUctuXumL0n/PIgdodaVPNCx7Lov2xfJfY3DLmni2p9d4S/Ajr5NM1+jdZqXfHIg39af5G+Zc/9lt27+w/wI3+TxmY+BpXEOZmXRpoqsrlOcnsl0kV6njV1DM1OrfUs9XIjbjAq9HehLeu6tg6Lpd2o6hcqqKo7tvvfovUrHzH41z+LtUc5ylVg1t+woT6Jeb9T080eOsri3UnXS51aZTL9TVv9L+KXqaUWGz2htK3tJE76/Qq98vK1j+yiXuJ9QACeK6AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADsx6bci+FNMJWWTkoxjFbttnBJtpJbt9yJt5ScFQ0qmvWtTqTz7Y70VyX91F+PxOarqmU0avcdlDRvrJUjb/8ADDarwHjcO8rM/UdQrjZqtns+vhSnNe6vXbvN15Uy24B0xfwy/nkc+dNkYctc2LfWdlaXr76Z5eVc1/YPTVv3Rnv/AK5FVmnkqaHtH8Vf7FzpaaOkuPZR6YZ9cm1Zc181tbf7D/Aq589y6qL8OvIshj2z7VlaltGTW+2/2lnMp9rGtXnB/gVbt6Wz/wATO/Z9qfeJ5e5HbUqu9F8fY4gAshUQAAAAfYxlJ7RTb8kgD4DeOD+WPEfEEK8mdKwMOXX2t62bXpHvZKvD/KPhrS3G3L9tqFy2e9r2in8ERdXeKWlXdc7K9EJejslXVpvNbhOq6EAafpGqag0sLT8rI38YVNr7TddN5Q8UZNcLL3i4sZLdxnNuUfikWAxMPEwoOOJj10prbaEdjtkyBn2kldpE1E+pY6bZWFuszlXy0ISq5KZnZTt1yiL8UqW/6nbDkq1/ea4mv4af+pMcmdcmcv8AztYv6k+R2/8AjtAn6V+ZC+TyZy4xbo1qmT8FKpr+ph8zlPxNSm6p4l6XgrNm/uJ8kzrkzcy+1acVRfgan7OUSpoip8StWfwTxRhLe7SL3HzhtL8DCZeHl4k+xlY11EvKyDj+JaxvxPJnYeJmVyry8aq+L71OKe53RbQOz32aEfLss3H3cmviVXBPGtct+HM+MpUUTwrX3SqfT7O4jziDlvrundqzFjDOpXc6+k9vgS9PdKefRFwviQdVZaun1VuU8DSgdl9NtFsqr6p1WR74zjs0dZI8SJVMaKAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAbvyz0T5xqscmyhWyrXaXaW8a/KT835InPElKbUp79e78iEOFuNpaZp1GnYemxlkSntKbe/bb7vrNn5h8c26fpq0rGcVqd1f6+VculCfgtvEgrjRzVT0ROH91LParhT0cSqqZX+6H3ntxTg26bXw3h2+1vVsbMiUXvGKW+0fjvt9hmuVnZhwNp+37Sm3/rZAc5SnNznJyk3u231ZPHLGW3BGnL+GX88jVcKVtLRtib1/c22msfWXB8z+ae6G0XS/UzX8LKxZK2yLF5Tf4ll7pJ0z6/ssrPe97pt+Mn+Jmwtxv/D3M7ULnsvj7HAAFhKkDuxMTKzLVViY119j/Zrg5P7jrrSdkU3sm1uyxnCWtcJaJj4unaRZDUc62uP6vEpW+/Z67vp9pxVtU6naitarlX+6ndQ0jal6o96NRP7oR7y+5c3ZlleXxBh3Y9Dl7tdm6c1/hXUm3SuE9C01dnG0zGitu/2a6/UYziDiujRaK9Q1GynD7+3jOcZze68l1/A1DN54adVHs42mXZU9vpt+zW/w6v7yt1TbhXasTTwUtNI620HdkXK9VQlx7JJRikl3JLbY6pv4kAapzo4iyIShh4mHib/tKLk/vNR1DjXinP8A/Ea1ltdekZ7L7jRDs3UuXMjkQ6JtqaViYjRV+iFn8vUMDGbjkZuNS/KdsYv72YfUeMeGcFfr9ZxN/KE+0/uKuXZGRdJyuvssk+9yk2dRJRbNRN/G9V+GCMl2rlVPu40T45LL3cw+EIQ7X6Zql6KMt/wMVkc1uE65dmNuVZ6xq/6lfQdLdn6VOa/M5nbT1i8k+X8k9Pmzww5bdnN28/ZL8z3YPMThTL7tR9g34WwcfzK7g9LYKVeGTy3aWsTjhfgWnxNS0/NSeHm49+63ShYm/sO6TKrUX3UTU6bZ1yXjGTRsui8fcSaZKK+eyya109nf7y2+Pgccuz7k1jd8yQg2oaukzMeRYFnCTNA4d5oaZmyjTqlMsKx9O2n2ofmjd8XLxsylXYuRXfW/2oSTREzUk0C4e0sFPXwVSZjdkx+ucP6PrNbjn4VdkvCaW0l9aI+1zlZJN2aPnJrwrv8AzRKkjg2bqeung0a74GiqtlNU6vbr1TRSuevaFqeiXqrUMaVe/wBGffGXwZjCx+s6bhathyxM6iN1T6pPvT815EQ8Z8DZmjdrKwnLKwl1bS96HxLHRXNk+Gv0cVG5WSSly+PvN+qGnAAlCDAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAOVc51zjOEnGUXumu9MWTnZNzsk5Sk92292ziAATpyze3BWB8JfzMgsnDlu9uC8Bekv5mRF5TMKefspYdm/wAy7y90NhzrHHCvnHvVbf3Fb5/TfxLFZ8v9hv8A/Tf4FdZ/Tl8TVZUw1/wN+0y9+P4+x8ABOFXB2Y992Narce2dU13ShLZnWADsvvuyJ9u+2dsvOcm2dYAAAAAAAAAAAAAAAAAMho+s6lpF6t0/Ltpfik+j+KMeDDmo5MKh6a9zF3mrhSXOEOZNOZZDE1uMMe2XRXrpB/FeBv8AC2u2tWVTjOElupRe6ZWQymj6/q2k2RnhZtsEv2G94v4oh6qzseu9Fov0LDRbQyxJuzpvJ15lhmddiU4uMkmmtmmjUeC+OMXWksTNUMbN26LfaFnw8n6G3MgZqeSB+69C109VFVM341yhGPMHghQ7ep6NU2u+2iK++P5EcNNNprZosjLqR5zB4Mjf7TVNJr2t+lbSv2vVepN2+45xHKvxK1d7NjM0CeaEYA+tNNpppro0z4ThVgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAATby5f+5uB8JfzMhImrl0/9z8H4S/mZFXf/Cnn7KWDZz8y7/190M9nN/MrtvGuX4FeZ/Tl8SwWc38yua/5b/Ar5P6b+Jqs6d1/wN+0q99nx9j4ACaKwAAAAAAAAAAAAAAAAAAAAAAAAAAAfYSlCSlCTjJPdNeBJfAPHKmoaZrNnvbpVXt9/pL8yMwaKinZO3dcdVJWSUr99i/yWRbTW6e6ficX1+BFvA/HEsNV6dq0nOhdK7u9w+PmiTqra7ao2VTjOElvGUXumvMq9TSvp3Ydw6l9oq+Osj3mceadDQOY3CMLYWavple1kV2rqorpJea9SMyxc0pRcX1T6NEN8w9DWkat7WiG2NkbyjsukZeKJi2VayJ2b+PIrd8tyRL20aYReJrAAJcrgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAJn5eS24QwfhL+ZkMEy8vn/ujg/CX8zIu6pmJPMn9nfzLvL3QzuU98a1P9x/gV/n9OXxJ/yHtj2P8Ahf4EAWf3kvizXaeDvgb9pPxRr5+xxABMFZAAAAByhCU32YRlJ+SW4BxBksXQdZyWvY6bkPfxcNvxMjTwVxDY1viRr/xzRrdNG3iqG9lNM/8AC1fka4Db6eX2szf6y3Ggv8TZksXlx/8Alaj9VcPzNLq2BP1HSy11blxuKR8CTI8udO/azsl/BL8jsr5d6Uku3lZMn49UjWtxgTmb0sdYv6fqReCVHy+0VrpZkr/Mdc+XelN+7l5SX1fkYS5QL1+R6WxVick+ZF4JOfLrTdumblL7PyOm3lxi/wDD1C5f4ooylwgXma1stYn6fqRuDfb+XGQt/Y6jW/JSgzw3cvtah9CzGs/ztG1KyBf1Gl1rq28Y1NQBncvhLX8bdywJzS8YNMxeRgZ2Ot78S+tecq2jc2RjuCnK+CVn4mqnwPMAD2agbpy84remWx07Pm3hzfuTf/Cf5Glg1SxNlarXJob6eokp5EkjXUsUpKUVKLTTW6a7mjHcQaVjaxp1mFkLpJbxl4xfg0aHy+4vePKvStTs/UPpVa39D0foSU3ut13PruiszQyUsvopfKWqir4V+SoQNremZGk6jZhZK96D6S8JLwaPCS9zB0D9L6a8iiO+XjpuH8S8URE002mtmu9FhpKhJ488+ZTLjQuo5lbyXgfAAdRHgAAAAAAAAAAAAAAAAAAAAAAAAAAAmTl+/wDdHC+Ev5mQ2THwA/8AdLC+Ev5mRt0/xJ5k/s7+Zd5e6Gdt61SXnFogK1bWzX8TJ9lu4teaNEx+X1c8uy3MzGq5TbjCtbdN/NnHb6iOFHK9SSvVHLUqzskzjPsR0e/B0fVM5pYuDfYn3Ps7L7WS1pvDWjafs6MKtzX7c12pfazLQSitopRXgkjokuifoaccOzr11ldjyIx03gDVL9pZdtWNF+H0pGfw+Xul1tPIyL7/ADW/ZX3G47jc4n3CZ/BcErDZqSPimfMweNwjw/Q944EJ/wCNuX4mVx8DBxo9mjEprS/dgkelRm1v2Xt5mM1PXNK07f51m0xl+7GXaf3Grenl5qp07lJTpwRDJp+XcN/qNRv4/wBFgvchkWvy7CSMfk8x6V/4bTJP/wBSz8j22hndyNL7vRs03vkb9uNyM7+Y2oyX6rBxa/tf4s8VnHuvy+jZTBelaN6WyVeKocr9oKdF0RVJZ39Bv6EO28Z8R2dP0jOC8oJI6rOLeIZ7drU7m16npLXJ1Q1LtFD/AKqTPuN/QhX+1Ov7t/pO/r6nFcTa8u7VMlf5x/xcnVDK7RQf6L9CbN/QbkLR4q4gT3/SmQ/jLc92Px3r9S2lbTav4q0FtcnJUMptFAvFqkt7jcjSHMjU+x2bMHDk/NJoyOFzHxJ9lZumzrfi6p7r7GanW2ZORvZfaVy4VcG9bnGcYTW04RkvJrc1WHHuhSl/9xHf96vu+8ymFxJouXsqtQpTfhN9n8TQtPMzVUU7W11LLoj0U553D2i5m/ttPp3f7UY7M1/P5e6ba5Sxci6hvuT95G30303R7VNsLF5xkmdm4ZUzR8FUxJQ0s/4moRRqfAus4u8qFDKgu7sPaX2M1zKxMrFn2cnHtpf8cWied2dWTjUZNbryKa7YvvUo7nbHdHJo9MkTPs9G7WJ2PMgQ3ngjjOWIoafqs3LH7q7e9w9H6Ge1TgbR8tudCnizf/LfT7DS+IOENU0p9uFbyqP361u18Udnb09U3ccRa0dbbndq1NPD3JfqsruqjbVONkJLdSi90yLuZWhfMM/9I48dsfIfvpLpGf8A1PLwbxTfot3zfJ7duHJ+9Dxh6oke9adxLolldVsbaLY9674vw+DOFsb6GbPFqks+WG7UytTR6cvH9iEQerVcG/Ts+3DyI7WVy2+K8GeUnEVFTKFRc1WrhQADJgAAAAAAAAAAAAAAAAAAAAAAAAExcv8A/wApYXwl/MyHSYuANo8I4TfRdmT/APcyNun+JPMntnlxUO8vdDPAwercVaNp28bMlXWL9ir3n+RqWqcwcyxyhgY0KYvulP3pfkRkVFNJwTHmWCoutLBlFdlfAkeycK4Odk4wiu9yeyRh87inQ8PdWZ0JyX7NfvMijUNX1LP6ZeZdYv3XLp9h4CQjtbf1qQs+0L1XETceZJOocw8OCawcSy1+ErH2V9hgszj3Wrt/Yexxk/3I7v7WamDsjo4WcGkZNdauXRX4Tw0Mjm67q+Z0ydRyJryc3sY+TcnvJtvzZ8B0IiJohwOc5y5VcgAGTyAAAAAAAAAAAAAAAAAAd+Pl5WPJSoyLa2v3ZNGa07jHXMR7PJ9vHytW/wB5rwPD42P/ABJk2xzyRascqEi6ZzDpm1HUcR1v9+p7r7DadM17SdRinjZtTl+7J7S+xkIn2MnF7xbT80cUluid+HQlYL7Ux6O7yE/jbdbPuIZ0rijWtNaVOZKcF07FnvR+82HT+Y+bVP8A2rBpsi1s+w9mcL7ZKi91ckxHtBTuTvoqG067wtpOqpynQqLn/wAWpJN/HzNRs0vX+EMmWZgTeTid80l3r+Jf1Pfhcw6JZDjlYc4Ut+7KL3aXqvE2LC4k0PPrahm1LddY2e70+s9p9phTdemW/M1qlBV9+J24/wCRp3FE8TifSo6vgRUczHjtkU/tdnz9UjSTJ6hfHB1vKlpeQ/ZduShKPc4vw+BjCXiajWoicCsVEqyv3l48/HxAANhoAAAAAAAAAAAAAAAAAAAAAAAAB7Vq2orBjgxy7Y48e6ClsjxAwqIvE9Nc5vBQ+r3YAMnkAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA//9k='

def _to_div(fig, first=False):
    return pio.to_html(fig, full_html=False,
        include_plotlyjs='cdn' if first else False,
        config={'displayModeBar': True, 'responsive': True})

# ── Aggregated stats: LF channels + HF derived channels ─
def agg_stats(arr, name):
    a = arr[np.isfinite(arr)]
    if len(a) == 0:
        return {f'{name}_max':'N/A', f'{name}_mean':'N/A',
                f'{name}_min':'N/A', f'{name}_std':'N/A'}
    return {f'{name}_max': f'{a.max():.4f}', f'{name}_mean': f'{a.mean():.4f}',
            f'{name}_min': f'{a.min():.4f}', f'{name}_std': f'{a.std():.4f}'}

lf_agg = {}
lf_agg.update(agg_stats(lf_df['inclination_deg'].values
    if 'inclination_deg' in lf_df.columns else np.array([np.nan]), 'inclination_deg'))
lf_agg.update(agg_stats(lf_df['temp_external_c'].values
    if 'temp_external_c' in lf_df.columns else np.array([np.nan]), 'temp_external_c'))
lf_agg.update(agg_stats(lf_df['temp_internal_c'].values
    if 'temp_internal_c' in lf_df.columns else np.array([np.nan]), 'temp_internal_c'))
lf_agg.update(agg_stats(lf_df['rpm_1hz'].values
    if 'rpm_1hz' in lf_df.columns else np.array([np.nan]), 'rpm_1hz'))

hf_agg = {}
hf_agg.update(agg_stats(rms_lateral, 'rms_lateral_g'))
hf_agg.update(agg_stats(rms_z,       'rms_axial_g'))
hf_agg.update(agg_stats(shock_max,   'shock_max_g'))
hf_agg.update(agg_stats(ssi,         'ssi'))
hf_agg.update(agg_stats(rpm_mean,    'rpm_mean'))

def sev_badge(val_str, metric):
    try: val = float(val_str)
    except: return '<span style="color:#8b92a5">N/A</span>'
    s = severity_label(val, metric) if metric in THRESH else 'NORMAL'
    col = {'CRITICAL':'#ef4444','ADVISORY':'#f59e0b','NORMAL':'#22c55e'}[s]
    return (f'<span style="background:{col};color:#000;font-weight:700;'
            f'font-size:10px;padding:2px 8px;border-radius:3px">{s}</span>')

# ── Build all report figures ────────────────────────────────────────────────────
def _make_ts():
    f = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=['Vibration RMS (g)', 'RPM & SSI', 'Temperature (°C)', 'Inclination (°)'])
    for arr, col, nm in [(rms_x,ICS['blue'],'RMS X'),(rms_y,ICS['teal'],'RMS Y'),
                          (rms_z,ICS['red'],'RMS Z'),(rms_lateral,ICS['amber'],'Lateral')]:
        f.add_trace(go.Scatter(x=t_ds, y=arr[::DS], name=nm, line=dict(color=col, width=1.2)), row=1, col=1)
    for yv, col in [(1.0,ICS['amber']),(3.0,ICS['red'])]:
        f.add_trace(go.Scatter(x=[t_ds[0],t_ds[-1]], y=[yv,yv], mode='lines',
            line=dict(color=col, dash='dash', width=0.8), showlegend=False, hoverinfo='skip'), row=1, col=1)
    rms_max_r = max(float(rms_lateral.max()), float(rms_z.max()), 0.1)
    for tr in [wh_trace, bb_trace, ss_trace]:
        if tr is not None:
            f.add_trace(scale_fill(tr, rms_max_r), row=1, col=1)
    f.add_trace(go.Scatter(x=t_ds, y=rpm_mean[::DS], name='RPM mean',
        line=dict(color=ICS['amber'], width=1.5)), row=2, col=1)
    f.add_trace(go.Scatter(x=t_ds, y=ssi[::DS]*100, name='SSI x100',
        line=dict(color=ICS['red'], width=1, dash='dot')), row=2, col=1)
    if ss_trace is not None:
        f.add_trace(scale_fill(ss_trace, max(float(rpm_mean.max()),1.0)), row=2, col=1)
    if HAS_TEMP:
        _t_ext = np.nan_to_num(temp_ext, nan=0.0)
        _t_int = np.nan_to_num(temp_int, nan=0.0)
        f.add_trace(go.Scatter(x=lf_t, y=_t_ext, name='Borehole Temp',
            line=dict(color=ICS['orange'], width=1.5),
            fill='tozeroy', fillcolor='rgba(249,115,22,0.06)'), row=3, col=1)
        if not np.all(_t_int == 0):
            f.add_trace(go.Scatter(x=lf_t, y=_t_int, name='Internal Temp',
                line=dict(color='#fb923c', width=1, dash='dot')), row=3, col=1)
        _tmax_ts = float(np.nanmax(_t_ext)) if np.any(_t_ext > 0) else 200.0
        for yv, col in [(140, ICS['amber']), (175, ICS['red'])]:
            f.add_trace(go.Scatter(x=[float(lf_t[0]), float(lf_t[-1])], y=[yv,yv], mode='lines',
                line=dict(color=col, dash='dot', width=0.8),
                showlegend=False, hoverinfo='skip'), row=3, col=1)
        if ot_trace is not None:
            f.add_trace(scale_fill(ot_trace, _tmax_ts), row=3, col=1)
        f.update_yaxes(range=[0, max(_tmax_ts * 1.15, 200)], row=3, col=1)
    if HAS_INC:
        f.add_trace(go.Scatter(x=lf_t, y=inclination, name='Inclination',
            line=dict(color=ICS['blue'], width=1.6),
            fill='tozeroy', fillcolor='rgba(59,130,246,0.08)'), row=4, col=1)
    else:
        f.add_annotation(text='❌  INCLINATION DATA UNAVAILABLE',
            x=0.5, y=0.12, xref='paper', yref='paper', showarrow=False,
            font=dict(size=12, color=ICS['red']))
    f.update_layout(title='Time-Series Overview', height=880, **ICS_LAYOUT)
    for r, lbl in [(1,'g-RMS'),(2,'RPM/SSI'),(3,'°C'),(4,'°')]:
        f.update_yaxes(title_text=lbl, row=r, col=1, **GRID_STYLE)
    f.update_xaxes(title_text='Elapsed Time (s)', row=4, col=1, **GRID_STYLE)
    return f

def _make_raw_corr():
    _rms_arr = [rms_x, rms_y, rms_z]
    _rms_cols = [ICS['teal'], '#a78bfa', ICS['orange']]
    _dis_map = [(wh_trace,'purple'),(bb_trace,'red'),(ss_trace,'amber')]
    _ax_cfg = [
        (ax_raw, ax, rms_x, 'rgba(59,130,246,0.4)', ICS['blue'],   ICS['teal'],   'X'),
        (ay_raw, ay, rms_y, 'rgba(20,184,166,0.4)', ICS['teal'],   '#a78bfa',     'Y'),
        (az_raw, az, rms_z, 'rgba(239,68,68,0.4)',  ICS['red'],    ICS['orange'], 'Z'),
    ]
    f = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=[
            'X-Axis: Raw vs Corrected + RMS  |  purple=Whirl  red=Bit-Bounce  amber=Stick-Slip',
            'Y-Axis: Raw vs Corrected + RMS  |  purple=Whirl  amber=Stick-Slip',
            'Z-Axis: Raw vs Corrected + RMS  |  red=Bit-Bounce',
        ])
    for i, (raw, corr, rms_a, raw_col, corr_col, rms_col, label) in enumerate(_ax_cfg, 1):
        f.add_trace(go.Scatter(x=t_ds, y=raw[::DS],  name=f'{label} Raw',
            line=dict(color=raw_col, width=0.7)), row=i, col=1)
        f.add_trace(go.Scatter(x=t_ds, y=corr[::DS], name=f'{label} Corrected',
            line=dict(color=corr_col, width=1.3)), row=i, col=1)
        f.add_trace(go.Scatter(x=t_ds, y=rms_a[::DS], name=f'{label} RMS',
            line=dict(color=rms_col, width=2.0)), row=i, col=1)
        _ymax = max(float(np.abs(raw).max()), float(rms_a.max()), 0.1)
        # Dysfunction shading per row
        if i == 1:   _shading = [wh_trace]
        elif i == 2: _shading = [wh_trace, ss_trace]
        else:        _shading = [bb_trace]
        for tr in _shading:
            if tr is not None:
                f.add_trace(scale_fill(tr, _ymax), row=i, col=1)
        f.update_yaxes(title_text='g', row=i, col=1, **GRID_STYLE)
    f.update_layout(title='Raw vs Temperature-Corrected Acceleration + RMS', height=720, **ICS_LAYOUT)
    f.update_xaxes(title_text='Elapsed Time (s)', row=3, col=1, **GRID_STYLE)
    return f

def _make_psd():
    from scipy.signal import welch as _welch
    NPERSEG2 = min(2048, N//4 or 512)
    def cpsd(sig): return _welch(sig-sig.mean(), fs=SAMPLE_RATE, nperseg=NPERSEG2)
    fx,px=cpsd(ax); fy,py=cpsd(ay); fz,pz=cpsd(az); fr,pr=cpsd(rpm_inst)
    f2=make_subplots(rows=2,cols=2,subplot_titles=[
        'Vibration PSD — X-Axis (Lateral)', 'Vibration PSD — Y-Axis (Lateral)',
        'Vibration PSD — Z-Axis (Axial)',    'RPM PSD — Torsional'])
    for fi,p,col,nm,r,c in [(fx,px,ICS['blue'],'X',1,1),(fy,py,ICS['teal'],'Y',1,2),
                              (fz,pz,ICS['red'],'Z',2,1),(fr,pr,ICS['purple'],'RPM',2,2)]:
        f2.add_trace(go.Scatter(x=fi, y=10*np.log10(p+1e-12), name=nm,
            line=dict(color=col, width=1.5)), row=r, col=c)
        f2.update_xaxes(range=[0,20], title_text='Frequency (Hz)', row=r, col=c, **GRID_STYLE)
        f2.update_yaxes(title_text='dB (g²/Hz)', row=r, col=c, **GRID_STYLE)
    f_rot = rpm_mean.mean()/60.0
    if f_rot > 0.01:
        for h in [1,2,3]:
            if h*f_rot < 20:
                f2.add_vline(x=h*f_rot, line_dash='dash', line_color=ICS['amber'],
                    annotation_text=f'{h}× RPM', row=2, col=1)
        f2.add_vrect(x0=0.1, x1=2.0, fillcolor=ICS['amber'], opacity=0.07,
            annotation_text='Stick-slip band', line_width=0, row=2, col=2)
    f2.update_layout(title='Power Spectral Density — Vibration & Torsional', height=560, **ICS_LAYOUT)
    return f2

def _make_rpm_hist():
    cmd = rpm_mean.mean()
    f3=go.Figure()
    f3.add_trace(go.Histogram(x=rpm_inst[rpm_inst>=0], nbinsx=80,
        name='RPM distribution', marker_color=ICS['purple'], opacity=0.75))
    for xv,col,lbl in [(cmd,ICS['amber'],f'Mean {cmd:.0f} RPM'),
                        (0,ICS['red'],'Stick (0 RPM)'),
                        (cmd*2,ICS['orange'],'Slip (2× RPM)')]:
        f3.add_vline(x=xv, line_dash='dash', line_color=col, annotation_text=lbl,
                     annotation_font_color=col)
    low_p = (rpm_inst < 0.15*cmd).sum()/max(len(rpm_inst),1)
    high_p = (rpm_inst > 1.8*cmd).sum()/max(len(rpm_inst),1)
    diag = ('BIMODAL — Severe Stick-Slip' if low_p>0.10 and high_p>0.05
            else 'Broadened — Mild Stick-Slip' if ssi.mean()>0.10
            else 'Single-peak — Healthy')
    dcol = ICS['red'] if 'Severe' in diag else ICS['amber'] if 'Mild' in diag else ICS['green']
    f3.add_annotation(text=diag, x=0.5, y=0.92, xref='paper', yref='paper',
        showarrow=False, font=dict(size=13, color=dcol), bgcolor='rgba(11,12,14,0.7)', borderpad=5)
    f3.update_layout(title='RPM Histogram — Stick-Slip Diagnostic',
        xaxis_title='Instantaneous RPM', yaxis_title='Count', height=380, **ICS_LAYOUT)
    f3.update_xaxes(**GRID_STYLE); f3.update_yaxes(**GRID_STYLE)
    return f3

def _make_bitbounce():
    f5 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=['Z-Axis Acceleration','Axial RMS & Peak Shock','Shock Count/s'])
    f5.add_trace(go.Scatter(x=t_ds, y=az[::DS], name='Accel Z',
        line=dict(color=ICS['red'], width=0.8)), row=1, col=1)
    z_max = float(np.abs(az).max())
    for tr in [bb_trace, ss_trace]:
        if tr is not None: f5.add_trace(scale_fill(tr, z_max), row=1, col=1)
    f5.add_trace(go.Scatter(x=t_ds, y=rms_z[::DS], name='Axial RMS',
        line=dict(color=ICS['amber'], width=1.5)), row=2, col=1)
    f5.add_trace(go.Scatter(x=t_ds, y=shock_max[::DS], name='Peak Shock',
        line=dict(color=ICS['red'], width=1, dash='dot')), row=2, col=1)
    rms_z_max = max(float(rms_z.max()), float(shock_max.max()), 1.0)
    for tr in [bb_trace, ss_trace]:
        if tr is not None: f5.add_trace(scale_fill(tr, rms_z_max), row=2, col=1)
    f5.add_trace(go.Bar(x=t_ds, y=shock_1s[::DS], name='Shock/s',
        marker_color=ICS['red'], opacity=0.7), row=3, col=1)
    f5.update_layout(title='Bit Bounce & Shock  |  red=Bit-Bounce  amber=Stick-Slip',
        height=680, **ICS_LAYOUT)
    f5.update_xaxes(title_text='Elapsed Time (s)', row=3, col=1, **GRID_STYLE)
    for r,l in [(1,'g'),(2,'g-RMS'),(3,'Count/s')]:
        f5.update_yaxes(title_text=l, row=r, col=1, **GRID_STYLE)
    return f5

def _make_whirl():
    f6 = make_subplots(rows=2, cols=2, subplot_titles=[
        'X vs Y Orbit (Lissajous)','Lateral RMS — X / Y / Total',
        'X & Y Corrected Signals', 'Lateral / Axial Ratio'])
    pts = min(5000, N)
    f6.add_trace(go.Scatter(x=ax[:pts], y=ay[:pts], mode='lines', name='XY orbit',
        line=dict(color='rgba(20,184,166,0.5)', width=0.8)), row=1, col=1)
    f6.add_trace(go.Scatter(x=[0], y=[0], mode='markers',
        marker=dict(color=ICS['red'],size=8,symbol='x'), name='Tool centre'), row=1, col=1)
    f6.update_xaxes(title_text='Accel X (g)', row=1, col=1, **GRID_STYLE)
    f6.update_yaxes(title_text='Accel Y (g)', row=1, col=1, **GRID_STYLE)
    for arr,col,nm in [(rms_x,ICS['blue'],'RMS X'),(rms_y,ICS['teal'],'RMS Y'),
                        (rms_lateral,ICS['amber'],'Lateral total')]:
        f6.add_trace(go.Scatter(x=t_ds,y=arr[::DS],name=nm,
            line=dict(color=col,width=1.5)), row=1, col=2)
    for yv,col in [(1.0,ICS['amber']),(3.0,ICS['red'])]:
        f6.add_trace(go.Scatter(x=[t_ds[0],t_ds[-1]],y=[yv,yv],mode='lines',
            line=dict(color=col,dash='dash',width=1),showlegend=False,hoverinfo='skip'), row=1, col=2)
    lat_max = max(float(rms_lateral.max()), 0.1)
    for tr in [wh_trace, bb_trace, ss_trace]:
        if tr is not None: f6.add_trace(scale_fill(tr, lat_max), row=1, col=2)
    f6.add_trace(go.Scatter(x=t_ds,y=ax[::DS],name='Accel X',
        line=dict(color=ICS['blue'],width=0.8)), row=2, col=1)
    f6.add_trace(go.Scatter(x=t_ds,y=ay[::DS],name='Accel Y',
        line=dict(color=ICS['teal'],width=0.8)), row=2, col=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        lat_ax = np.where(rms_z>0.01, rms_lateral/(rms_z+0.01), rms_lateral)
    f6.add_trace(go.Scatter(x=t_ds,y=lat_ax[::DS],name='Lat/Axial',
        line=dict(color=ICS['purple'],width=1.5)), row=2, col=2)
    f6.add_trace(go.Scatter(x=[t_ds[0],t_ds[-1]],y=[2.0,2.0],mode='lines',
        line=dict(color=ICS['amber'],dash='dash',width=1),
        name='Whirl threshold',hoverinfo='skip'), row=2, col=2)
    if wh_trace is not None:
        f6.add_trace(scale_fill(wh_trace, max(float(lat_ax.max()),2.1)), row=2, col=2)
    for r in [2]: f6.update_xaxes(title_text='Elapsed Time (s)', row=r, col=1, **GRID_STYLE)
    for r in [2]: f6.update_xaxes(title_text='Elapsed Time (s)', row=r, col=2, **GRID_STYLE)
    f6.update_layout(title='Lateral Whirl Detection  |  purple=Whirl  red=Bit-Bounce  amber=Stick-Slip',
        height=680, **ICS_LAYOUT)
    return f6

def _make_spectro():
    from scipy.signal import spectrogram as _sp
    FS2=SAMPLE_RATE; NP=min(2048,N//4 or 512); NOV=int(NP*0.875); MXFQ=min(6.0,FS2/2-0.1)
    def stft(sig):
        fi,ts,Sxx=_sp(sig.astype(float),fs=FS2,nperseg=min(NP,len(sig)//4 or NP),
            noverlap=min(NOV,(len(sig)//4 or NP)-1),window='hann',scaling='spectrum')
        amp=np.sqrt(np.abs(Sxx)); mask=fi<=MXFQ
        return fi[mask], ts+t[0], amp[mask]
    sigs=[('Axial',az),('Lateral',lateral_g),('Tangential',tangential_g),('RPM',rpm_inst)]
    f9=make_subplots(rows=4,cols=1,shared_xaxes=True,vertical_spacing=0.04,
        subplot_titles=[f'{nm} Spectrogram' for nm,_ in sigs])
    fr_rot=rpm_mean.mean()/60.0
    for ri,(nm,sig) in enumerate(sigs,1):
        fh,th,amp=stft(sig); vmax=np.percentile(amp,99) or 1.0
        f9.add_trace(go.Heatmap(z=amp,x=th,y=fh,colorscale=AMBER_SCALE,
            zmin=0,zmax=vmax,colorbar=dict(len=0.22,y=1.0-(ri-1)*0.255,
            yanchor='top',thickness=12,tickfont=dict(size=9,color=ICS['text'])),
            name=nm,showscale=True),row=ri,col=1)
        f9.update_yaxes(title_text=f'{nm} (Hz)',range=[0,MXFQ],row=ri,col=1,**GRID_STYLE)
        if fr_rot>0.01:
            for h in [1,2,3]:
                if h*fr_rot<MXFQ:
                    f9.add_hline(y=h*fr_rot,line_dash='dash',line_color=ICS['amber'],
                        line_width=0.8,row=ri,col=1)
    f9.update_xaxes(title_text='Elapsed Time (s)',row=4,col=1,**GRID_STYLE)
    f9.update_layout(title='STFT Spectrograms — RPM harmonic lines in amber',
        height=900,margin=dict(r=120),**ICS_LAYOUT)
    return f9

def _make_inclination():
    if not HAS_INC:
        f10 = go.Figure()
        f10.add_annotation(text='❌  INCLINATION DATA UNAVAILABLE',
            x=0.5, y=0.5, xref='paper', yref='paper', showarrow=False,
            font=dict(size=16, color=ICS['red']))
        f10.update_layout(title='Inclination vs Time — DATA UNAVAILABLE',
            height=340, **ICS_LAYOUT)
        f10.update_xaxes(visible=False); f10.update_yaxes(visible=False)
        return f10
    inc_rate = np.gradient(inclination, lf_t)
    f10 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
        subplot_titles=['Inclination vs Time',
                        'Inclination Rate of Change (°/s)'])
    f10.add_trace(go.Scatter(x=lf_t, y=inclination, name='Inclination (°)',
        line=dict(color=ICS['blue'], width=1.6),
        fill='tozeroy', fillcolor='rgba(59,130,246,0.08)'), row=1, col=1)
    f10.add_trace(go.Scatter(x=lf_t, y=inc_rate, name='Inc. Rate (°/s)',
        line=dict(color=ICS['amber'], width=1.2)), row=2, col=1)
    f10.add_hline(y=0, line_dash='dot', line_color=ICS['grid'], row=2, col=1)
    f10.update_layout(
        title=f'Inclination vs Time  |  Range: {inclination.min():.1f}° — {inclination.max():.1f}°',
        height=560, **ICS_LAYOUT)
    f10.update_xaxes(title_text='Elapsed Time (s)', row=2, col=1, **GRID_STYLE)
    f10.update_yaxes(title_text='Inclination (°)', row=1, col=1, **GRID_STYLE)
    f10.update_yaxes(title_text='Rate (°/s)',     row=2, col=1, **GRID_STYLE)
    return f10

print('  Rendering figures...')
figs = [
    ('Time-Series Overview',             _make_ts()),
    ('Raw vs Corrected Acceleration',     _make_raw_corr()),
    ('Power Spectral Density',            _make_psd()),
    ('RPM Histogram',                     _make_rpm_hist()),
    ('Bit Bounce & Shock Events',         _make_bitbounce()),
    ('Lateral Whirl Detection',           _make_whirl()),
    ('STFT Spectrograms',                 _make_spectro()),
    ('Inclination vs Time',               _make_inclination()),
]
print(f'  {len(figs)} figures rendered.')

divs = [(title, _to_div(fig, first=(i==0))) for i,(title,fig) in enumerate(figs)]

# ── Stats table ────────────────────────────────────────────────────────────────
def _stat_row(label, stat_prefix, metric=None):
    row  = f'<tr style="border-bottom:1px solid #1f232d">'
    row += f'<td style="padding:7px 14px;color:#8b92a5;font-family:Space Mono,monospace;font-size:10px">{label}</td>'
    for suffix in ['max','mean','min','std']:
        key = f'{stat_prefix}_{suffix}'
        val = lf_agg.get(key, hf_agg.get(key, 'N/A'))
        row += f'<td style="padding:7px 14px;text-align:center;font-family:Space Mono,monospace;font-size:11px;color:#e8eaf0">{val}</td>'
    badge = sev_badge(hf_agg.get(f'{stat_prefix}_max','N/A'), metric) if metric else ''
    row += f'<td style="padding:7px 14px;text-align:center">{badge}</td></tr>'
    return row

stats_html = (
    _stat_row('Lateral RMS (g)',   'rms_lateral_g',  'lateral_rms') +
    _stat_row('Axial RMS (g)',     'rms_axial_g',    'axial_rms')   +
    _stat_row('Peak Shock (g)',    'shock_max_g',    'peak_shock')  +
    _stat_row('SSI',               'ssi',            'ssi')         +
    _stat_row('RPM (mean)',        'rpm_mean',        None)          +
    _stat_row('Inclination (°)',   'inclination_deg', None)          +
    _stat_row('Borehole Temp (°C)','temp_external_c','temperature') +
    _stat_row('Internal Temp (°C)','temp_internal_c','temperature') +
    _stat_row('RPM 1 Hz',          'rpm_1hz',        None)
)

# ── Dysfunction bars ───────────────────────────────────────────────────────────
def _dysbars():
    bars = ''
    for nm,p,col in [
        ('Stick-Slip',    pct(stick_slip_flag), '#f59e0b'),
        ('Bit Bounce',    pct(bit_bounce_flag), '#ef4444'),
        ('Lateral Whirl', pct(whirl_flag),      '#8b5cf6'),
        ('Over-Temp',     pct(over_temp_flag),  '#f97316'),
    ]:
        bars += (f'<div style="margin:5px 0;display:flex;align-items:center;gap:8px">'
                 f'<span style="min-width:130px;color:#8b92a5;font-size:10px;font-family:Space Mono,monospace">{nm}</span>'
                 f'<div style="flex:1;background:#1f232d;border-radius:3px;height:14px">'
                 f'<div style="width:{max(p,0.5):.1f}%;height:100%;background:{col};border-radius:3px;opacity:0.85"></div></div>'
                 f'<span style="min-width:42px;text-align:right;color:#e8eaf0;font-size:10px;font-family:Space Mono,monospace">{p:.1f}%</span>'
                 f'</div>')
    return bars

ts_now   = datetime.now().strftime('%d %b %Y  %H:%M:%S')
file_hf  = os.path.basename(HF_PATH)
file_lf  = os.path.basename(LF_PATH)
run_dur  = f'{t[-1]:.0f}s ({t[-1]/60:.1f} min)'

# Per-panel divs with HIDE/SHOW
plot_panels = ''.join(
    f'<div class="panel" style="margin-bottom:18px">'
    f'<div class="panel-header">'
    f'<div class="panel-title"><div class="dot"></div>{title}</div>'
    f'<button onclick="togglePanel(this)" style="background:none;border:1px solid rgba(255,255,255,0.1);'
    f'color:#8b92a5;font-size:10px;padding:3px 10px;border-radius:4px;cursor:pointer;'
    f'font-family:Space Mono,monospace">HIDE</button>'
    f'</div>'
    f'<div class="panel-body" style="padding:8px">{div_html}</div>'
    f'</div>'
    for title, div_html in divs
)

# ── CSS ─────────────────────────────────────────────────────────────────────────
CSS = """
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
body{background:#0b0c0e;color:#e8eaf0;font-family:'DM Sans',sans-serif;font-size:15px;line-height:1.6;overflow-x:hidden}
body::before{content:'';position:fixed;inset:0;
  background-image:linear-gradient(rgba(255,255,255,0.022) 1px,transparent 1px),
    linear-gradient(90deg,rgba(255,255,255,0.022) 1px,transparent 1px);
  background-size:40px 40px;pointer-events:none;z-index:0}
.header{position:relative;z-index:10;border-bottom:1px solid rgba(255,255,255,0.07);
  background:rgba(17,19,24,0.98);padding:0 36px}
.header-top{display:flex;align-items:center;justify-content:space-between;
  padding:16px 0;border-bottom:1px solid rgba(255,255,255,0.07)}
.logo-img{height:52px;width:auto;object-fit:contain;flex-shrink:0}
.logo-text{font-family:'Syne',sans-serif;font-size:13px;font-weight:700;
  letter-spacing:0.15em;text-transform:uppercase;color:#8b92a5}
.logo-text span{color:#f59e0b}
.pill{font-family:'Space Mono',monospace;font-size:10px;color:#8b92a5;
  background:#1f232d;border:1px solid rgba(255,255,255,0.07);border-radius:4px;padding:3px 10px}
.header-title{padding:18px 0 14px}
.page-title{font-family:'Syne',sans-serif;font-size:26px;font-weight:800;letter-spacing:-0.02em;color:#e8eaf0}
.page-title span{color:#f59e0b}
.main{position:relative;z-index:1;padding:24px 36px;max-width:1440px;margin:0 auto}
.kpi-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:24px}
.kpi{background:#181b22;border:1px solid rgba(255,255,255,0.07);border-radius:9px;
  padding:16px;position:relative;overflow:hidden}
.kpi::before{content:'';position:absolute;top:0;left:0;right:0;height:2px;background:#f59e0b}
.kpi-label{font-family:'Space Mono',monospace;font-size:9px;letter-spacing:0.12em;
  text-transform:uppercase;color:#8b92a5;margin-bottom:6px}
.kpi-value{font-family:'Syne',sans-serif;font-size:26px;font-weight:800;line-height:1;color:#f59e0b}
.kpi-sub{font-size:10px;color:#8b92a5;margin-top:3px;font-family:'Space Mono',monospace}
.two-col{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:16px}
.panel{background:#181b22;border:1px solid rgba(255,255,255,0.07);border-radius:9px;overflow:hidden}
.panel-header{display:flex;align-items:center;justify-content:space-between;
  padding:12px 16px;border-bottom:1px solid rgba(255,255,255,0.07);background:#1f232d}
.panel-title{font-family:'Syne',sans-serif;font-size:12px;font-weight:700;
  color:#e8eaf0;display:flex;align-items:center;gap:7px}
.dot{width:6px;height:6px;border-radius:50%;background:#f59e0b}
.stats-table{width:100%;border-collapse:collapse}
.stats-table th{font-family:'Space Mono',monospace;font-size:9px;letter-spacing:0.1em;
  text-transform:uppercase;color:#8b92a5;text-align:center;padding:9px 14px;
  border-bottom:1px solid rgba(255,255,255,0.07);background:#1f232d}
.stats-table th:first-child{text-align:left}
.footer{border-top:1px solid rgba(255,255,255,0.07);padding:16px 36px;display:flex;
  align-items:center;justify-content:space-between;position:relative;z-index:1}
.footer-text{font-family:'Space Mono',monospace;font-size:10px;color:#8b92a5}
::-webkit-scrollbar{width:5px}::-webkit-scrollbar-thumb{background:#1f232d;border-radius:3px}
@media(max-width:1100px){.kpi-grid{grid-template-columns:repeat(2,1fr)}.two-col{grid-template-columns:1fr}}
"""

html_out = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>ZerdaLabs GeoTorpedo Analytics Suite — {file_hf}</title>
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@400;700;800&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet">
<style>{CSS}</style>
</head>
<body>
<header class="header">
  <div class="header-top">
    <div style="display:flex;align-items:center;gap:14px">
      <img src="{LOGO_DATA_URI}" class="logo-img" alt="ZerdaLabs">
      <div>
        <div class="logo-text"><span>ZerdaLabs</span></div>
        <div style="font-size:10px;color:#8b92a5;font-family:Space Mono,monospace;margin-top:2px">
          GeoTorpedo &middot; Downhole Drilling Analytics
        </div>
      </div>
    </div>
    <div style="display:flex;gap:8px;flex-wrap:wrap">
      <div class="pill">HF: {file_hf}</div>
      <div class="pill">LF: {file_lf}</div>
      <div class="pill">{run_dur}</div>
      <div class="pill">{ts_now}</div>
    </div>
  </div>
  <div class="header-title">
    <h1 class="page-title">ZerdaLabs GeoTorpedo <span>Analytics Suite</span></h1>
    <p style="font-size:12px;color:#8b92a5;margin-top:3px">
      Vibration &middot; RPM &middot; Inclination &middot; Temperature &middot; 1 Hz Statistics
    </p>
  </div>
</header>
<main class="main">
{META_BANNER_HTML}
<div class="kpi-grid">
  <div class="kpi">
    <div class="kpi-label">Peak Lateral RMS</div>
    <div class="kpi-value">{float(rms_lateral.max()):.2f}g</div>
    <div class="kpi-sub">critical &gt; 3g</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Peak Shock</div>
    <div class="kpi-value">{float(shock_max.max()):.0f}g</div>
    <div class="kpi-sub">critical &gt; 50g</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Mean RPM</div>
    <div class="kpi-value">{float(rpm_mean.mean()):.0f}</div>
    <div class="kpi-sub">1s rolling mean</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Max SSI</div>
    <div class="kpi-value">{float(ssi.max()):.3f}</div>
    <div class="kpi-sub">critical &gt; 0.50</div>
  </div>
</div>
<div class="two-col">
  <div class="panel">
    <div class="panel-header"><div class="panel-title"><div class="dot"></div>Aggregated Statistics</div></div>
    <div style="padding:0;overflow-x:auto">
      <table class="stats-table">
        <thead><tr>
          <th style="text-align:left">Channel</th>
          <th>Max</th><th>Mean</th><th>Min</th><th>Std Dev</th><th>Status</th>
        </tr></thead>
        <tbody>{stats_html}</tbody>
      </table>
    </div>
  </div>
  <div class="panel">
    <div class="panel-header"><div class="panel-title"><div class="dot"></div>Dysfunction Occurrence (% of run)</div></div>
    <div style="padding:14px">{_dysbars()}</div>
  </div>
</div>
{plot_panels}
</main>
<footer class="footer">
  <div class="footer-text">ZerdaLabs / GeoTorpedo Analytics Suite &middot; Downhole Drilling Analytics v3.2</div>
  <div class="footer-text">Generated: {ts_now}</div>
</footer>
<script>
function togglePanel(btn) {{
  var body = btn.closest('.panel').querySelector('.panel-body');
  if (body.style.display === 'none') {{ body.style.display = ''; btn.textContent = 'HIDE'; }}
  else {{ body.style.display = 'none'; btn.textContent = 'SHOW'; }}
}}
</script>
</body>
</html>"""

base     = os.path.splitext(HF_PATH)[0]
ts_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = f'{base}_report_{ts_stamp}.html'

with open(out_path, 'w', encoding='utf-8') as fh:
    fh.write(html_out)

size_kb = os.path.getsize(out_path)/1024
print(f'\nHTML report saved:')
print(f'   {out_path}')
print(f'   {size_kb:.0f} KB  |  {len(figs)} interactive charts')
print(f'\nOpen in Chrome or Edge to view.')


## Cell 20 — Live Grafana-Style Dashboard (Standalone Browser)

Serialises all derived BHA channels to a **self-contained `.html` file** and opens it in your default browser.

- No VS Code dependency — works in any browser (Chrome, Edge, Firefox)
- All data embedded as JSON — no server required
- Rolling window, speed control, dysfunction detection — identical to previous cell
- File written to same directory as the notebook

**Usage:** Run the cell → `bha_live_dashboard.html` opens automatically.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 20 — Live BHA Dashboard → Standalone HTML file + auto-open in browser
# Writes bha_live_dashboard.html next to the notebook, then opens it.
# No VS Code iframe, no server — pure browser HTML/JS.
# ═══════════════════════════════════════════════════════════════════════════════

import json, math, os, webbrowser
import numpy as np
from pathlib import Path
from IPython.display import display, HTML

# ── 1. Resolve output path ────────────────────────────────────────────────────
try:
    _nb_dir = Path(__file__).parent
except NameError:
    _nb_dir = Path.cwd()           # Jupyter/VS Code: cwd is the notebook dir

OUT_HTML = _nb_dir / "bha_live_dashboard.html"

# ── 2. Decimate to 10 Hz for smooth JS playback ───────────────────────────────
LIVE_HZ   = 10
STRIDE    = max(1, SAMPLE_RATE // LIVE_HZ)

t_live       = t[::STRIDE]
rms_x_live   = rms_x[::STRIDE]
rms_y_live   = rms_y[::STRIDE]
rms_z_live   = rms_z[::STRIDE]
rms_lat_live = rms_lateral[::STRIDE]
rpm_live     = rpm_inst[::STRIDE]
rpm_m_live   = rpm_mean[::STRIDE]
ssi_live     = ssi[::STRIDE]
shk_live     = shock_max[::STRIDE]

if HAS_TEMP:
    tmp_live = np.interp(t_live, lf_t, temp_ext)
else:
    tmp_live = np.full(len(t_live), float('nan'))

if HAS_INC:
    inc_live = np.interp(t_live, lf_t, inclination)
else:
    inc_live = np.full(len(t_live), float('nan'))

N_LIVE   = len(t_live)
WIN_DEF  = 30   # default rolling window seconds
WIN_FRAMES_DEF = WIN_DEF * LIVE_HZ

# ── 3. Compact JSON serialisation ─────────────────────────────────────────────
def _f3(arr):
    return [round(float(v), 3) if math.isfinite(float(v)) else None for v in arr]

payload = {
    't':       _f3(t_live),
    'rms_x':   _f3(rms_x_live),
    'rms_y':   _f3(rms_y_live),
    'rms_z':   _f3(rms_z_live),
    'rms_lat': _f3(rms_lat_live),
    'rpm':     _f3(rpm_live),
    'rpm_m':   _f3(rpm_m_live),
    'ssi':     _f3(ssi_live),
    'shk':     _f3(shk_live),
    'tmp':     _f3(tmp_live),
    'inc':     _f3(inc_live),
}

meta = {
    'sample_rate': int(SAMPLE_RATE),
    'live_hz':     LIVE_HZ,
    'n_frames':    N_LIVE,
    'duration_s':  round(float(t_live[-1]), 2) if N_LIVE else 0,
    'has_temp':    bool(HAS_TEMP),
    'has_inc':     bool(HAS_INC),
    'win_default': WIN_DEF,
    'win_frames_default': WIN_FRAMES_DEF,
    'thresh':      THRESH,
}

DATA_JSON = json.dumps(payload, separators=(',', ':'))
META_JSON = json.dumps(meta,    separators=(',', ':'))

# ── 4. Build the full standalone HTML ─────────────────────────────────────────
HTML_CONTENT = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>ZerdaLabs GeoTorpedo — Live Monitor</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@400;600;700;800&display=swap" rel="stylesheet">
<style>
:root{{
  --bg0:#0b0c0e;--bg1:#111318;--bg2:#181b22;--bg3:#1f232d;--bg4:#262b38;
  --amber:#f59e0b;--amber-dim:#92600a;--amber-glow:rgba(245,158,11,0.08);
  --teal:#14b8a6;--red:#ef4444;--green:#22c55e;--blue:#3b82f6;
  --purple:#8b5cf6;--orange:#f97316;
  --text:#e8eaf0;--muted:#8b92a5;--dim:#4a5168;
  --border:rgba(255,255,255,0.07);--grid:rgba(255,255,255,0.035);
  --fm:'Space Mono',monospace;--fd:'Syne',sans-serif;
}}
*,*::before,*::after{{box-sizing:border-box;margin:0;padding:0}}
html,body{{height:100%;overflow:hidden;background:var(--bg0);color:var(--text);font-family:var(--fm);font-size:11px}}

/* background grid texture */
body::before{{
  content:'';position:fixed;inset:0;
  background-image:linear-gradient(var(--grid) 1px,transparent 1px),
                   linear-gradient(90deg,var(--grid) 1px,transparent 1px);
  background-size:32px 32px;pointer-events:none;z-index:0;
}}

/* ─── HEADER ─── */
.hdr{{
  position:relative;z-index:10;
  display:flex;align-items:center;justify-content:space-between;
  padding:0 16px;height:44px;
  background:rgba(17,19,24,0.97);border-bottom:1px solid var(--border);
}}
.logo-group{{display:flex;align-items:center;gap:10px}}
.logo-img{{height:30px;width:auto;object-fit:contain;flex-shrink:0}}
.hex{{
  width:26px;height:26px;background:var(--amber);flex-shrink:0;
  clip-path:polygon(50% 0%,100% 25%,100% 75%,50% 100%,0% 75%,0% 25%);
}}
.brand{{font-family:var(--fd);font-size:13px;font-weight:800;letter-spacing:0.1em;text-transform:uppercase}}
.brand span{{color:var(--amber)}}
.sub{{font-size:9px;color:var(--muted);margin-top:1px;letter-spacing:0.06em}}
.pills{{display:flex;gap:6px;align-items:center;flex-wrap:wrap}}
.pill{{font-size:10px;color:var(--muted);background:var(--bg3);border:1px solid var(--border);border-radius:3px;padding:2px 8px}}
.pill.live{{border-color:var(--amber-dim);color:var(--amber);background:var(--amber-glow)}}
.pill.live::before{{content:'● ';animation:blink 1s step-end infinite}}
@keyframes blink{{50%{{opacity:0}}}}

/* ─── CONTROL BAR ─── */
.ctrl{{
  position:relative;z-index:10;
  display:flex;align-items:center;gap:8px;
  padding:0 16px;height:36px;
  background:var(--bg2);border-bottom:1px solid var(--border);
}}
.btn{{
  font-family:var(--fm);font-size:10px;font-weight:700;letter-spacing:0.06em;
  text-transform:uppercase;padding:3px 12px;border-radius:3px;
  border:none;cursor:pointer;transition:opacity .15s;flex-shrink:0;
}}
.btn:hover{{opacity:.75}}
.btn-start{{background:var(--green);color:#050}}
.btn-stop{{background:var(--red);color:#fff}}
.btn-reset{{background:var(--bg4);color:var(--muted);border:1px solid var(--border)}}
.sep{{width:1px;height:18px;background:var(--border)}}
.ctrl-lbl{{color:var(--muted);font-size:9px;letter-spacing:0.06em;flex-shrink:0}}
input[type=range]{{
  -webkit-appearance:none;height:3px;background:var(--bg4);border-radius:2px;outline:none;
}}
input[type=range]::-webkit-slider-thumb{{
  -webkit-appearance:none;width:11px;height:11px;border-radius:50%;cursor:pointer;
}}
#sld-spd{{width:90px}} #sld-spd::-webkit-slider-thumb{{background:var(--amber)}}
#sld-win{{width:100px}} #sld-win::-webkit-slider-thumb{{background:var(--teal)}}
.sval{{min-width:32px;text-align:left}}
#sv-spd{{color:var(--amber)}} #sv-win{{color:var(--teal)}}
.prog-wrap{{flex:1;height:3px;background:var(--bg4);border-radius:2px;overflow:hidden;min-width:60px}}
.prog-bar{{height:100%;background:var(--amber);width:0%;transition:width .1s linear;border-radius:2px}}
.t-disp{{color:var(--amber);min-width:90px;text-align:right;font-size:11px}}

/* ─── DYSFUNCTION STRIP ─── */
.dys-strip{{
  position:relative;z-index:10;
  display:flex;align-items:center;gap:6px;
  padding:0 16px;height:28px;
  background:var(--bg1);border-bottom:1px solid var(--border);
}}
.dys{{
  padding:1px 9px;border-radius:3px;font-size:9px;font-weight:700;
  letter-spacing:0.08em;text-transform:uppercase;
  opacity:0.3;transition:opacity .25s,box-shadow .25s;
  border:1px solid transparent;
}}
.dys.on{{opacity:1}}
.dys-ss{{background:rgba(245,158,11,.1);color:var(--amber);border-color:var(--amber-dim)}}
.dys-ss.on{{box-shadow:0 0 8px rgba(245,158,11,.35)}}
.dys-bb{{background:rgba(239,68,68,.1);color:var(--red);border-color:rgba(239,68,68,.35)}}
.dys-bb.on{{box-shadow:0 0 8px rgba(239,68,68,.35)}}
.dys-wh{{background:rgba(139,92,246,.1);color:var(--purple);border-color:rgba(139,92,246,.35)}}
.dys-wh.on{{box-shadow:0 0 8px rgba(139,92,246,.35)}}
.dys-ot{{background:rgba(249,115,22,.1);color:var(--orange);border-color:rgba(249,115,22,.35)}}
.dys-ot.on{{box-shadow:0 0 8px rgba(249,115,22,.35)}}
.dys-strip-r{{margin-left:auto;color:var(--dim);font-size:9px}}

/* ─── LAYOUT ─── */
.layout{{
  position:relative;z-index:1;
  display:grid;
  grid-template-columns:repeat(12,1fr);
  grid-template-rows:52px 1fr 1fr 1fr;
  gap:5px;padding:5px;
  height:calc(100vh - 44px - 36px - 28px);
  overflow:hidden;
}}

/* ─── KPI STRIP ─── */
.kpi-strip{{
  grid-column:span 12;
  display:grid;grid-template-columns:repeat(6,1fr);gap:5px;
}}
.kpi{{
  background:var(--bg2);border:1px solid var(--border);border-radius:5px;
  padding:5px 10px;display:flex;flex-direction:column;justify-content:center;
  position:relative;overflow:hidden;
}}
.kpi::before{{content:'';position:absolute;top:0;left:0;right:0;height:2px}}
.k-amber::before{{background:var(--amber)}}
.k-red::before{{background:var(--red)}}
.k-teal::before{{background:var(--teal)}}
.k-purple::before{{background:var(--purple)}}
.k-orange::before{{background:var(--orange)}}
.k-green::before{{background:var(--green)}}
.kpi-lbl{{font-size:8px;letter-spacing:0.1em;text-transform:uppercase;color:var(--muted);line-height:1}}
.kpi-val{{font-family:var(--fd);font-size:20px;font-weight:800;line-height:1.1;margin-top:2px}}
.kpi-sev{{font-size:8px;font-weight:700;letter-spacing:0.07em;margin-top:1px}}
.sev-N{{color:var(--green)}} .sev-A{{color:var(--amber)}} .sev-C{{color:var(--red)}}

/* ─── CHART PANELS ─── */
.cp{{
  background:var(--bg2);border:1px solid var(--border);border-radius:5px;
  display:flex;flex-direction:column;overflow:hidden;
}}
.cp-h{{
  display:flex;align-items:center;justify-content:space-between;
  padding:4px 10px;border-bottom:1px solid var(--border);background:var(--bg3);flex-shrink:0;
}}
.cp-t{{
  display:flex;align-items:center;gap:5px;
  font-size:9px;font-weight:700;letter-spacing:0.09em;text-transform:uppercase;
}}
.dot{{width:6px;height:6px;border-radius:50%;flex-shrink:0}}
.cp-leg{{display:flex;gap:8px}}
.leg{{display:flex;align-items:center;gap:3px;font-size:8px;color:var(--muted)}}
.leg-line{{width:14px;height:2px;border-radius:1px}}
.cp-body{{flex:1;position:relative;min-height:0}}
canvas{{position:absolute;inset:0;width:100%!important;height:100%!important}}
.unavail{{
  display:flex;align-items:center;justify-content:center;
  height:100%;color:var(--dim);font-size:10px;letter-spacing:0.05em;
}}

/* expand-to-fullscreen button */
.cp-r{{display:flex;align-items:center;gap:10px;margin-left:auto}}
.cp-exp{{
  background:var(--bg4);border:1px solid var(--border);color:var(--muted);
  font-family:var(--fm);font-size:9px;font-weight:700;letter-spacing:0.06em;
  padding:2px 7px;border-radius:3px;cursor:pointer;flex-shrink:0;
  transition:background .15s,color .15s,border-color .15s;
}}
.cp-exp:hover{{background:var(--amber);color:#050;border-color:var(--amber)}}

/* fullscreen state — panel takes over viewport */
.cp.fullscreen{{
  position:fixed;inset:8px;z-index:1000;
  grid-column:unset!important;grid-row:unset!important;
  box-shadow:0 0 0 9999px rgba(0,0,0,0.7),0 0 30px rgba(245,158,11,0.25);
  border-color:var(--amber-dim);
}}
body.has-fs{{overflow:hidden}}

/* grid placements */
.r2-vib  {{grid-column:span 7;grid-row:2}}
.r2-rpm  {{grid-column:span 5;grid-row:2}}
.r3-ssi  {{grid-column:span 4;grid-row:3}}
.r3-la   {{grid-column:span 4;grid-row:3}}
.r3-shk  {{grid-column:span 4;grid-row:3}}
.r4-tmp  {{grid-column:span 6;grid-row:4}}
.r4-inc  {{grid-column:span 6;grid-row:4}}
</style>
</head>
<body>

<!-- HEADER -->
<header class="hdr">
  <div class="logo-group">
    <img src="{LOGO_DATA_URI}" class="logo-img" alt="ZerdaLabs">
    <div>
      <div class="brand"><span>ZerdaLabs</span> GeoTorpedo Analytics Suite — Live Monitor</div>
      <div class="sub">GeoTorpedo · Downhole Drilling Data · Real-Time Data</div>
    </div>
  </div>
  <div class="pills">
    <div class="pill live" id="p-live">LIVE</div>
    <div class="pill" id="p-sr">—</div>
    <div class="pill" id="p-dur">—</div>
    <div class="pill" id="p-nf">—</div>
    <div class="pill" id="p-ts">t = 0.000 s</div>
  </div>
</header>

<!-- CONTROLS -->
<div class="ctrl">
  <button class="btn btn-start" onclick="ctrlStart()">▶ Start</button>
  <button class="btn btn-stop"  onclick="ctrlStop()">⏹ Stop</button>
  <button class="btn btn-reset" onclick="ctrlReset()">↺ Reset</button>
  <div class="sep"></div>
  <span class="ctrl-lbl">Speed</span>
  <input type="range" id="sld-spd" min="1" max="20" value="5" oninput="setSpeed(this.value)">
  <span class="sval" id="sv-spd">0.5×</span>
  <div class="sep"></div>
  <span class="ctrl-lbl">Window</span>
  <input type="range" id="sld-win" min="5" max="120" value="30" step="5" oninput="setWindow(this.value)">
  <span class="sval" id="sv-win">30 s</span>
  <div class="sep"></div>
  <div class="prog-wrap"><div class="prog-bar" id="prog-bar"></div></div>
  <div class="t-disp" id="t-disp">t = 0.000 s</div>
</div>

<!-- DYSFUNCTION STRIP -->
<div class="dys-strip">
  <div class="dys dys-ss" id="d-ss">⚡ Stick-Slip</div>
  <div class="dys dys-bb" id="d-bb">↕ Bit Bounce</div>
  <div class="dys dys-wh" id="d-wh">↻ Lateral Whirl</div>
  <div class="dys dys-ot" id="d-ot">🌡 Over-Temp</div>
  <div class="dys-strip-r">Rolling window: <span id="win-lbl">30 s</span></div>
</div>

<!-- DASHBOARD GRID -->
<div class="layout">

  <!-- Row 1: KPI strip -->
  <div class="kpi-strip">
    <div class="kpi k-amber">
      <div class="kpi-lbl">Lateral RMS</div>
      <div class="kpi-val" id="kv-lat">—</div>
      <div class="kpi-sev" id="ks-lat">&nbsp;</div>
    </div>
    <div class="kpi k-red">
      <div class="kpi-lbl">Axial RMS</div>
      <div class="kpi-val" id="kv-ax">—</div>
      <div class="kpi-sev" id="ks-ax">&nbsp;</div>
    </div>
    <div class="kpi k-teal">
      <div class="kpi-lbl">Inst RPM</div>
      <div class="kpi-val" id="kv-rpm" style="color:var(--teal)">—</div>
      <div class="kpi-sev" id="ks-rpm" style="color:var(--muted)">&nbsp;</div>
    </div>
    <div class="kpi k-purple">
      <div class="kpi-lbl">Stick-Slip Index</div>
      <div class="kpi-val" id="kv-ssi">—</div>
      <div class="kpi-sev" id="ks-ssi">&nbsp;</div>
    </div>
    <div class="kpi k-orange">
      <div class="kpi-lbl">Peak Shock</div>
      <div class="kpi-val" id="kv-shk">—</div>
      <div class="kpi-sev" id="ks-shk">&nbsp;</div>
    </div>
    <div class="kpi k-green">
      <div class="kpi-lbl">Temperature</div>
      <div class="kpi-val" id="kv-tmp">—</div>
      <div class="kpi-sev" id="ks-tmp">&nbsp;</div>
    </div>
  </div>

  <!-- Row 2 -->
  <div class="cp r2-vib">
    <div class="cp-h">
      <div class="cp-t"><div class="dot" style="background:var(--amber)"></div>Vibration RMS — X / Y / Z</div>
      <div class="cp-leg">
        <div class="leg"><div class="leg-line" style="background:var(--amber)"></div>X</div>
        <div class="leg"><div class="leg-line" style="background:var(--teal)"></div>Y</div>
        <div class="leg"><div class="leg-line" style="background:var(--red)"></div>Z</div>
        <div class="leg"><div class="leg-line" style="background:rgba(255,255,255,.2);border-top:1px dashed rgba(255,255,255,.3)"></div>1 g</div>
      </div>
    </div>
    <div class="cp-body"><canvas id="cv-vib"></canvas></div>
  </div>
  <div class="cp r2-rpm">
    <div class="cp-h">
      <div class="cp-t"><div class="dot" style="background:var(--teal)"></div>Rotational Speed (RPM)</div>
      <div class="cp-leg">
        <div class="leg"><div class="leg-line" style="background:var(--teal);opacity:.7"></div>Inst</div>
        <div class="leg"><div class="leg-line" style="background:var(--amber)"></div>Mean</div>
      </div>
    </div>
    <div class="cp-body"><canvas id="cv-rpm"></canvas></div>
  </div>

  <!-- Row 3 -->
  <div class="cp r3-ssi">
    <div class="cp-h">
      <div class="cp-t"><div class="dot" style="background:var(--purple)"></div>Stick-Slip Index (SSI)</div>
      <div class="cp-leg">
        <div class="leg"><div class="leg-line" style="background:var(--amber);opacity:.6;border-top:1px dashed var(--amber)"></div>0.1</div>
        <div class="leg"><div class="leg-line" style="background:var(--red);opacity:.6;border-top:1px dashed var(--red)"></div>0.5</div>
      </div>
    </div>
    <div class="cp-body"><canvas id="cv-ssi"></canvas></div>
  </div>
  <div class="cp r3-la">
    <div class="cp-h">
      <div class="cp-t"><div class="dot" style="background:var(--amber)"></div>Lateral vs Axial g-RMS</div>
      <div class="cp-leg">
        <div class="leg"><div class="leg-line" style="background:var(--amber)"></div>Lateral</div>
        <div class="leg"><div class="leg-line" style="background:var(--red)"></div>Axial Z</div>
      </div>
    </div>
    <div class="cp-body"><canvas id="cv-la"></canvas></div>
  </div>
  <div class="cp r3-shk">
    <div class="cp-h">
      <div class="cp-t"><div class="dot" style="background:var(--orange)"></div>Peak Shock (g)</div>
      <div class="cp-leg">
        <div class="leg"><div class="leg-line" style="background:var(--red);opacity:.6;border-top:1px dashed var(--red)"></div>50 g</div>
      </div>
    </div>
    <div class="cp-body"><canvas id="cv-shk"></canvas></div>
  </div>

  <!-- Row 4 -->
  <div class="cp r4-tmp" id="panel-tmp">
    <div class="cp-h">
      <div class="cp-t"><div class="dot" style="background:var(--orange)"></div>Downhole Temperature</div>
      <div class="cp-leg">
        <div class="leg"><div class="leg-line" style="background:var(--orange)"></div>Ext °C</div>
        <div class="leg"><div class="leg-line" style="background:var(--amber);opacity:.6;border-top:1px dashed var(--amber)"></div>140°C</div>
        <div class="leg"><div class="leg-line" style="background:var(--red);opacity:.6;border-top:1px dashed var(--red)"></div>175°C</div>
      </div>
    </div>
    <div class="cp-body"><canvas id="cv-tmp"></canvas></div>
  </div>
  <div class="cp r4-inc" id="panel-inc">
    <div class="cp-h">
      <div class="cp-t"><div class="dot" style="background:var(--blue)"></div>Inclination (deg from vertical)</div>
    </div>
    <div class="cp-body"><canvas id="cv-inc"></canvas></div>
  </div>

</div><!-- /layout -->

<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<script>
// ─── EMBEDDED DATA ────────────────────────────────────────────────────────────
const D    = {DATA_JSON};
const META = {META_JSON};
const N    = META.n_frames;
const TH   = META.thresh;

// ─── STATE ────────────────────────────────────────────────────────────────────
let idx      = 0;
let running  = false;
let timer    = null;
let spd      = 0.5;      // frames per tick (fractional ok)
let spdAcc   = 0;        // sub-frame accumulator
let winF     = META.win_frames_default;

// ─── CHART.JS DEFAULTS ───────────────────────────────────────────────────────
Chart.defaults.animation = false;
Chart.defaults.color     = '#4a5168';
Chart.defaults.font.family = "'Space Mono',monospace";
Chart.defaults.font.size   = 9;

const GC = '#rgba(255,255,255,0.035)';
const TO = {{
  backgroundColor:'#1f232d',borderColor:'#262b38',borderWidth:1,
  titleColor:'#e8eaf0',bodyColor:'#8b92a5',padding:6,
  titleFont:{{family:"'Space Mono',monospace",size:9}},
  bodyFont:{{family:"'Space Mono',monospace",size:9}},
}};
function ax(overrides={{}}) {{
  return {{
    ticks:{{color:'#4a5168',maxTicksLimit:5}},
    grid:{{color:'rgba(255,255,255,0.04)'}},
    border:{{color:'rgba(255,255,255,0.05)'}},
    ...overrides
  }};
}}

function mkLine(id, sets, yOpt={{}}) {{
  const ctx = document.getElementById(id);
  if (!ctx) return null;
  return new Chart(ctx, {{
    type:'line',
    data:{{labels:[],datasets:sets.map(s=>Object.assign({{tension:0,pointRadius:0,borderWidth:1.5}},s))}},
    options:{{
      responsive:false, animation:false, normalized:true,
      plugins:{{legend:{{display:false}},tooltip:{{...TO,mode:'index',intersect:false}}}},
      scales:{{x:ax({{ticks:{{color:'#4a5168',maxTicksLimit:7}}}}),y:ax(yOpt)}}
    }}
  }});
}}

// Build charts
const cvVib = mkLine('cv-vib',[
  {{borderColor:'#f59e0b',label:'X RMS'}},
  {{borderColor:'#14b8a6',label:'Y RMS'}},
  {{borderColor:'#ef4444',label:'Z RMS'}},
  {{borderColor:'rgba(255,255,255,0.18)',borderDash:[4,3],label:'1 g',borderWidth:1}},
]);
const cvRpm = mkLine('cv-rpm',[
  {{borderColor:'rgba(20,184,166,0.6)',borderWidth:1,label:'Inst RPM'}},
  {{borderColor:'#f59e0b',borderWidth:2,label:'Mean RPM'}},
]);
const cvSsi = mkLine('cv-ssi',[
  {{borderColor:'#8b5cf6',fill:true,backgroundColor:'rgba(139,92,246,0.07)',label:'SSI'}},
  {{borderColor:'rgba(245,158,11,0.5)',borderDash:[4,3],borderWidth:1,label:'0.1'}},
  {{borderColor:'rgba(239,68,68,0.5)',borderDash:[4,3],borderWidth:1,label:'0.5'}},
],{{suggestedMax:0.7}});
const cvLa = mkLine('cv-la',[
  {{borderColor:'#f59e0b',label:'Lateral'}},
  {{borderColor:'#ef4444',label:'Axial Z'}},
]);
const cvShk = mkLine('cv-shk',[
  {{borderColor:'#f97316',fill:true,backgroundColor:'rgba(249,115,22,0.06)',borderWidth:1,label:'Shock'}},
  {{borderColor:'rgba(239,68,68,0.5)',borderDash:[4,3],borderWidth:1,label:'50 g'}},
]);
const cvTmp = mkLine('cv-tmp',[
  {{borderColor:'#f97316',fill:true,backgroundColor:'rgba(249,115,22,0.05)',label:'Temp °C'}},
  {{borderColor:'rgba(245,158,11,0.5)',borderDash:[4,3],borderWidth:1,label:'140°C'}},
  {{borderColor:'rgba(239,68,68,0.5)',borderDash:[4,3],borderWidth:1,label:'175°C'}},
]);
const cvInc = mkLine('cv-inc',[
  {{borderColor:'#3b82f6',fill:true,backgroundColor:'rgba(59,130,246,0.06)',label:'Inclination'}},
]);
const ALL = [cvVib,cvRpm,cvSsi,cvLa,cvShk,cvTmp,cvInc];

// Resize all canvases to fill their containers
function resizeAll() {{
  ALL.forEach(c=>{{
    if(!c) return;
    const el = c.canvas;
    const p  = el.parentElement;
    el.width  = p.clientWidth;
    el.height = p.clientHeight;
    c.resize();
  }});
}}
window.addEventListener('resize', resizeAll);
setTimeout(resizeAll, 100);

// ─── FULLSCREEN EXPAND/COLLAPSE ─────────────────────────────────────────────────
function toggleFullscreen(panel, btn) {{
  const isFs = panel.classList.toggle('fullscreen');
  document.body.classList.toggle('has-fs', !!document.querySelector('.cp.fullscreen'));
  btn.textContent = isFs ? '✕ Close' : '⛶ Expand';
  btn.title = isFs ? 'Close fullscreen (Esc)' : 'Expand to fullscreen';
  setTimeout(resizeAll, 50);
}}
document.querySelectorAll('.cp').forEach(panel => {{
  const hdr = panel.querySelector('.cp-h');
  if (!hdr) return;
  const btn = document.createElement('button');
  btn.className = 'cp-exp';
  btn.type = 'button';
  btn.textContent = '⛶ Expand';
  btn.title = 'Expand to fullscreen';
  btn.onclick = (e) => {{ e.stopPropagation(); toggleFullscreen(panel, btn); }};
  // wrap existing right-side legend + button together so legend stays visible
  const existingLeg = hdr.querySelector('.cp-leg');
  const right = document.createElement('div');
  right.className = 'cp-r';
  if (existingLeg) right.appendChild(existingLeg);
  right.appendChild(btn);
  hdr.appendChild(right);
}});
document.addEventListener('keydown', (e) => {{
  if (e.key === 'Escape') {{
    const fs = document.querySelector('.cp.fullscreen');
    if (fs) {{
      const btn = fs.querySelector('.cp-exp');
      toggleFullscreen(fs, btn);
    }}
  }}
}});

// Header pills
document.getElementById('p-sr').textContent  = `Fs = ${{META.sample_rate}} Hz`;
document.getElementById('p-dur').textContent  = `${{META.duration_s.toFixed(1)}} s`;
document.getElementById('p-nf').textContent   = `${{N.toLocaleString()}} frames`;

// ─── SEVERITY ─────────────────────────────────────────────────────────────────
function sev(v, m) {{
  if(v===null||isNaN(v)) return 'N';
  const t=TH[m]; if(!t) return 'N';
  return v>=t.critical?'C':v>=t.advisory?'A':'N';
}}
const sevTxt = s=>s==='C'?'CRITICAL':s==='A'?'ADVISORY':'NORMAL';
const sevClr = s=>s==='C'?'#ef4444':s==='A'?'#f59e0b':'#22c55e';
function setKpi(vid,sid,v,fmt,metric) {{
  const ev=document.getElementById(vid), es=document.getElementById(sid);
  if(v===null||isNaN(v)){{ev.textContent='—';ev.style.color='';return;}}
  ev.textContent = fmt(v);
  const s=sev(v,metric);
  ev.style.color=sevClr(s);
  es.textContent=sevTxt(s);
  es.className='kpi-sev sev-'+s;
}}

// ─── FRAME UPDATE ─────────────────────────────────────────────────────────────
function frame() {{
  if(idx>=N){{ctrlStop();return;}}
  const i  = idx;
  const i0 = Math.max(0, i - winF + 1);

  const tW  = D.t.slice(i0, i+1).map(v=>v!==null?v.toFixed(2):'');
  const sl  = (k)=> D[k]?D[k].slice(i0,i+1):[];
  const pts = (k)=> sl(k).map((v,j)=>v!==null?{{x:tW[j],y:v}}:{{x:tW[j],y:null}});
  const th  = (v) => tW.map(x=>{{return{{x,y:v}}}});

  function upd(ch, ...series) {{
    if(!ch) return;
    ch.data.labels = tW;
    series.forEach((s,si)=>{{ if(ch.data.datasets[si]) ch.data.datasets[si].data=s; }});
    ch.update('none');
  }}

  upd(cvVib, pts('rms_x'), pts('rms_y'), pts('rms_z'), th(1.0));
  upd(cvRpm, pts('rpm'),   pts('rpm_m'));
  upd(cvSsi, pts('ssi'),   th(0.1), th(0.5));
  upd(cvLa,  pts('rms_lat'), pts('rms_z'));
  upd(cvShk, pts('shk'),  th(50));
  upd(cvTmp, pts('tmp'),  th(140), th(175));
  upd(cvInc, pts('inc'));

  // KPIs
  const lat=D.rms_lat[i], axl=D.rms_z[i], rpm=D.rpm[i],
        ssi=D.ssi[i],     shk=D.shk[i],   tmp=D.tmp[i];
  setKpi('kv-lat','ks-lat',lat, v=>v.toFixed(3)+' g','lateral_rms');
  setKpi('kv-ax', 'ks-ax', axl, v=>v.toFixed(3)+' g','axial_rms');
  setKpi('kv-ssi','ks-ssi',ssi, v=>v.toFixed(3),     'ssi');
  setKpi('kv-shk','ks-shk',shk, v=>v.toFixed(1)+' g','peak_shock');
  setKpi('kv-tmp','ks-tmp',tmp, v=>isNaN(v)?'—':v.toFixed(1)+' °C','temperature');
  document.getElementById('kv-rpm').textContent = rpm!==null?rpm.toFixed(1):'—';
  document.getElementById('ks-rpm').textContent = rpm!==null?rpm.toFixed(1)+' RPM':'—';

  // Dysfunctions
  tog('d-ss', ssi!==null && ssi>TH.ssi.critical);
  tog('d-bb', axl!==null && axl>TH.axial_rms.critical && rpm!==null && rpm>10);
  tog('d-wh', lat!==null && lat>TH.lateral_rms.critical);
  tog('d-ot', tmp!==null && !isNaN(tmp) && tmp>TH.temperature.advisory);

  // Header
  const ts = D.t[i]??0;
  const ts3 = ts.toFixed(3);
  document.getElementById('p-ts').textContent   = `t = ${{ts3}} s`;
  document.getElementById('t-disp').textContent = `t = ${{ts.toFixed(2)}} s`;
  document.getElementById('prog-bar').style.width = `${{(i/(Math.max(N-1,1))*100).toFixed(2)}}%`;

  // Advance by speed
  spdAcc += spd;
  const adv = Math.floor(spdAcc);
  spdAcc -= adv;
  idx += Math.max(1, adv);
}}

function tog(id, on) {{
  document.getElementById(id).classList.toggle('on', on);
}}

// ─── CONTROLS ─────────────────────────────────────────────────────────────────
function ctrlStart() {{
  if(running) return;
  if(idx>=N) ctrlReset();
  running=true;
  document.getElementById('p-live').style.opacity='1';
  timer=setInterval(frame, 100);
}}
function ctrlStop() {{
  running=false;
  clearInterval(timer); timer=null;
  document.getElementById('p-live').style.opacity='0.35';
}}
function ctrlReset() {{
  ctrlStop(); idx=0; spdAcc=0;
  ALL.forEach(c=>{{if(c){{c.data.labels=[];c.data.datasets.forEach(d=>d.data=[]);c.update('none');}}}});
  document.getElementById('prog-bar').style.width='0%';
  document.getElementById('t-disp').textContent='t = 0.000 s';
  document.getElementById('p-ts').textContent='t = 0.000 s';
}};
function setSpeed(v) {{
  spd = v*0.5;
  document.getElementById('sv-spd').textContent = spd.toFixed(1)+'×';
}}
function setWindow(v) {{
  winF = parseInt(v)*META.live_hz;
  document.getElementById('sv-win').textContent  = v+' s';
  document.getElementById('win-lbl').textContent = v+' s';
}}

// Unavailable overlays
if(!META.has_temp) {{
  const p=document.getElementById('panel-tmp');
  if(p) p.innerHTML='<div class="unavail">❌ TEMPERATURE DATA UNAVAILABLE</div>';
}}
if(!META.has_inc) {{
  const p=document.getElementById('panel-inc');
  if(p) p.innerHTML='<div class="unavail">❌ INCLINATION DATA UNAVAILABLE</div>';
}}

document.getElementById('p-live').style.opacity='0.35';
</script>
</body>
</html>"""

# ── 5. Write file ─────────────────────────────────────────────────────────────
with open(OUT_HTML, 'w', encoding='utf-8') as _f:
    _f.write(HTML_CONTENT)

_sz_kb = OUT_HTML.stat().st_size / 1024
print(f"Written: {OUT_HTML}")
print(f"Size:    {_sz_kb:.1f} KB")
print(f"Frames:  {N_LIVE:,} @ {LIVE_HZ} Hz  |  Duration: {t_live[-1]:.1f} s")

# ── 6. Open in default browser ────────────────────────────────────────────────
webbrowser.open(OUT_HTML.as_uri())
print("Opened in browser — press ▶ Start in the dashboard to begin replay.")
